# STIR-Net V1 — 20 comprehensive spatial failure localization

This notebook is deliberately **diagnostic-heavy and training-free**.

The goal is to localize the spatial failure as far as possible before spending implementation-oriented Codex tokens.

Notebook 19 established that, at the final checkpoint:

- source-9 internal boundaries are poorly predicted;
- dense center peaks recover only part of the cells;
- E2/D1 features are weakly cell-specific;
- oracle GT centers do **not** rescue coarse instance masks;
- five extra cached query steps do not rescue the merge.

The unresolved questions are now:

1. **Was the spatial representation ever good?**
2. **If it degraded, at which curriculum transition did that happen?**
3. **Was degradation caused by the spatial features or only the dense boundary/center heads?**
4. **Does the query decoder fail even when fed the best earlier spatial checkpoint?**
5. **Can the decoder's mask-feature space express different cells at all?**
6. **Is the `max_spatial_tokens` cap destroying the spatial resolution needed to separate touching cells?**

There is **no overfit loop**, **no optimizer**, and **no native-resolution rendering** in this notebook.

Most work is a handful of forward passes and small closed-form diagnostic probes.

---

## Diagnostic map

```text
checkpoint trajectory
        │
        ├── parameter drift
        │
        ├── pure-spatial dense boundary / center quality
        │
        ├── E2 / D1 / D0 frozen-feature probes
        │
        ├── dense-head × feature checkpoint swap matrix
        │
        ├── decoder token-cap resolution audit
        │
        ├── oracle mask-vector expressivity probe
        │
        └── query-builder × query-decoder checkpoint swaps
                     │
                     ▼
             automatic localization report
```

The notebook uses GT only for **diagnostic counterfactuals/probes**. None of those are proposed as production inference logic.

In [ ]:
from pathlib import Path
from dataclasses import replace
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.coordinates import (
    feature_grid_coordinates_um,
    resize_label_map_nearest,
)
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
)
from learned.stirnet.model.query_decoder import _cap_feature_tokens
from learned.stirnet.model.types import TemporalState
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

TARGET_STEPS = [30, 50, 70, 80, 85]
STAGE_BY_STEP = {
    30: "spatial_dense_end",
    50: "temporal_dense_end",
    70: "query_bootstrap_end",
    80: "native_bootstrap_end",
    85: "joint_end",
}

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

PRIMARY_RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "15_hierarchical_temporal_memory"
)

FALLBACK_RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "20_spatial_failure_localization"
)
RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook 20 requires CUDA."
    )

device = torch.device("cuda")

cfg = _reduced_config()
cfg.curriculum.enabled = True
cfg.curriculum.spatial_dense_steps = 30
cfg.curriculum.temporal_dense_steps = 20
cfg.curriculum.query_bootstrap_steps = 20
cfg.curriculum.native_bootstrap_steps = 10

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Output     :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Max query spatial tokens:", cfg.decoder.max_spatial_tokens)

# 1. Discover same-run curriculum checkpoints

Checkpoint payload metadata is used instead of trusting filenames.

The notebook will **not silently mix checkpoints from different runs**.

In [ ]:
EXPECTED_NAMES = {
    30: [
        "checkpoint_spatial_dense.pt",
        "step30_spatial_dense.pt",
    ],
    50: [
        "checkpoint_temporal_dense.pt",
        "step50_temporal_dense.pt",
    ],
    70: [
        "checkpoint_query_bootstrap.pt",
        "step70_query.pt",
        "step70_query_bootstrap.pt",
    ],
    80: [
        "checkpoint_native_bootstrap.pt",
        "step80_native.pt",
        "step80_native_bootstrap.pt",
    ],
    85: [
        "checkpoint_joint.pt",
        "step85_joint.pt",
    ],
}

def checkpoint_step(path):
    try:
        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
        step = int(
            payload.get(
                "step",
                -1,
            )
        )
        del payload
        return step
    except Exception as exc:
        print(
            "Skipping unreadable:",
            path.name,
            repr(exc),
        )
        return -1

def scan_run(run_dir):
    run_dir = Path(
        run_dir
    )

    if not run_dir.exists():
        return {}

    candidates = sorted(
        set(
            list(
                run_dir.glob(
                    "*.pt"
                )
            )
            + list(
                run_dir.glob(
                    "*.pth"
                )
            )
        )
    )

    found = {}

    for path in candidates:
        step = checkpoint_step(
            path
        )

        if step in TARGET_STEPS:
            found.setdefault(
                step,
                [],
            ).append(
                path
            )

    chosen = {}

    for step, paths in (
        found.items()
    ):
        selected = None

        for expected in EXPECTED_NAMES[
            step
        ]:
            matching = [
                p
                for p in paths
                if p.name
                == expected
            ]

            if matching:
                selected = (
                    matching[0]
                )
                break

        if selected is None:
            selected = sorted(
                paths,
                key=lambda p: (
                    len(
                        p.name
                    ),
                    p.name,
                ),
            )[0]

        chosen[
            step
        ] = selected

    return chosen

primary = scan_run(
    PRIMARY_RUN_DIR
)

if len(primary) >= 2:
    CHECKPOINT_RUN = (
        PRIMARY_RUN_DIR
    )
    CHECKPOINTS = primary
else:
    fallback = scan_run(
        FALLBACK_RUN_DIR
    )

    if len(fallback) < 2:
        raise RuntimeError(
            "Could not locate at least two curriculum-boundary checkpoints "
            "inside one run directory."
        )

    CHECKPOINT_RUN = (
        FALLBACK_RUN_DIR
    )
    CHECKPOINTS = (
        fallback
    )

AVAILABLE_STEPS = sorted(
    CHECKPOINTS
)

checkpoint_df = pd.DataFrame([
    {
        "step": step,
        "stage": STAGE_BY_STEP[
            step
        ],
        "filename": CHECKPOINTS[
            step
        ].name,
        "path": str(
            CHECKPOINTS[
                step
            ]
        ),
    }
    for step in AVAILABLE_STEPS
])

print(
    "Using checkpoint run:",
    CHECKPOINT_RUN,
)
display(
    checkpoint_df
)

missing = [
    step
    for step in TARGET_STEPS
    if step not in CHECKPOINTS
]

if missing:
    print(
        "Missing same-run checkpoints:",
        missing,
    )
    print(
        "The diagnostic will continue with available stages."
    )

# 2. Load the real scene and identify source-9 GT cells

In [ ]:
batch, sample = build_real_batch(
    DATA_DIR
)
target = batch[
    "targets"
][0]

def prepare_device_batch(
    cpu_batch,
):
    result = {}

    for key, value in (
        cpu_batch.items()
    ):
        if key == "targets":
            result[key] = value

        elif key == "spatial_inputs":
            result[key] = (
                value.to(
                    device=device,
                    dtype=AMP_DTYPE,
                    non_blocking=True,
                )
            )

        elif key == "instance_labels":
            result[key] = (
                value.to(
                    device=device,
                    dtype=torch.int32,
                    non_blocking=True,
                )
            )

        else:
            result[key] = (
                move_to_device(
                    value,
                    device,
                )
            )

    return result

b = prepare_device_batch(
    batch
)

current_labels_cpu = (
    batch[
        "instance_labels"
    ][0]
    .detach()
    .cpu()
    .numpy()
)

gt_label_map_cpu = (
    torch.as_tensor(
        target[
            "label_map"
        ]
    )
    .detach()
    .cpu()
    .numpy()
)

source_mask_cpu = (
    current_labels_cpu
    == SOURCE_ID
)

overlap_ids, overlap_counts = (
    np.unique(
        gt_label_map_cpu[
            source_mask_cpu
        ],
        return_counts=True,
    )
)

positive = (
    overlap_ids > 0
)

source_gt_ids = (
    overlap_ids[
        positive
    ].astype(int)
)

source_gt_overlap = (
    overlap_counts[
        positive
    ].astype(int)
)

order = np.argsort(
    -source_gt_overlap
)

source_gt_ids = (
    source_gt_ids[
        order
    ]
)

source_gt_overlap = (
    source_gt_overlap[
        order
    ]
)

target_ids_cpu = (
    torch.as_tensor(
        target["ids"]
    )
    .detach()
    .cpu()
    .long()
)

id_to_target_row = {
    int(gt_id): row
    for row, gt_id
    in enumerate(
        target_ids_cpu.tolist()
    )
}

if "centers_um" in target:
    all_gt_centers_um = (
        torch.as_tensor(
            target[
                "centers_um"
            ]
        ).float()
    )
else:
    all_gt_centers_um = (
        torch.as_tensor(
            target[
                "centers_cellscale"
            ]
        ).float()
        * float(
            batch[
                "dref_um"
            ][0]
        )
    )

source_gt_rows = torch.tensor(
    [
        id_to_target_row[
            int(gt_id)
        ]
        for gt_id
        in source_gt_ids
    ],
    dtype=torch.long,
)

source_gt_centers_um = (
    all_gt_centers_um[
        source_gt_rows
    ]
    .detach()
    .cpu()
    .float()
)

dref_um = float(
    batch[
        "dref_um"
    ][0]
)

print("Scene:", sample)
print(
    "Source-9 GT count:",
    len(
        source_gt_ids
    ),
)
print(
    "dref_um:",
    dref_um,
)

display(
    pd.DataFrame({
        "gt_id": source_gt_ids,
        "source9_overlap_voxels": (
            source_gt_overlap
        ),
        "z_um": (
            source_gt_centers_um[
                :, 0
            ].numpy()
        ),
        "y_um": (
            source_gt_centers_um[
                :, 1
            ].numpy()
        ),
        "x_um": (
            source_gt_centers_um[
                :, 2
            ].numpy()
        ),
    })
)

if len(
    source_gt_ids
) < 2:
    raise RuntimeError(
        "Source 9 is not a multi-cell component."
    )

## Internal GT-boundary helper

At each feature resolution we downsample the **GT label map first**, then recompute inter-instance boundaries. This avoids losing thin boundaries through nearest-neighbour downsampling of a precomputed binary boundary mask.

In [ ]:
def internal_instance_boundary_np(
    labels,
):
    labels = np.asarray(
        labels
    )

    boundary = np.zeros_like(
        labels,
        dtype=bool,
    )

    for axis in range(3):
        left = [
            slice(None)
        ] * 3
        right = [
            slice(None)
        ] * 3

        left[
            axis
        ] = slice(
            0,
            -1,
        )
        right[
            axis
        ] = slice(
            1,
            None,
        )

        a = labels[
            tuple(left)
        ]
        c = labels[
            tuple(right)
        ]

        edge = (
            (a > 0)
            & (c > 0)
            & (a != c)
        )

        boundary[
            tuple(left)
        ] |= edge
        boundary[
            tuple(right)
        ] |= edge

    return boundary

def labels_at_shape(
    shape,
):
    gt = resize_label_map_nearest(
        torch.from_numpy(
            gt_label_map_cpu.astype(
                np.int32
            )
        ),
        shape,
    ).numpy()

    current = resize_label_map_nearest(
        torch.from_numpy(
            current_labels_cpu.astype(
                np.int32
            )
        ),
        shape,
    ).numpy()

    source = (
        current
        == SOURCE_ID
    )

    internal = (
        internal_instance_boundary_np(
            gt
        )
        & source
    )

    return (
        gt,
        current,
        source,
        internal,
    )

# 3. Static query-decoder resolution audit

Before examining learning, quantify how aggressively each query-decoder level is pooled by `max_spatial_tokens`.

If the capped lattice has only a handful of tokens per cell, no attention or query cleverness can recover boundaries that are no longer spatially resolved.

In [ ]:
@torch.no_grad()
def spatial_shapes_from_model(
    model,
):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        acquisition = (
            model.acquisition(
                b["spacing_um"],
                b["dref_um"],
            )
        )

        pyramid = (
            model.encoder(
                b[
                    "spatial_inputs"
                ],
                b["spacing_um"],
                acquisition,
                b.get(
                    "spatial_padding_mask"
                ),
            )
        )

        e3 = (
            pyramid.features[
                3
            ]
        )

        e2 = (
            model.decoder
            .decode_to_e2(
                e3,
                pyramid,
                acquisition,
            )
        )

        d1, _, _ = (
            model.decoder
            .decode_from_e2(
                e2,
                pyramid,
                acquisition,
            )
        )

    return (
        [
            e3,
            e2,
            d1,
        ],
        [
            pyramid.spacings_um[
                3
            ],
            pyramid.spacings_um[
                2
            ],
            pyramid.spacings_um[
                1
            ],
        ],
    )

shape_model = StirNet(
    cfg
).to(
    device
)

load_checkpoint(
    CHECKPOINTS[
        AVAILABLE_STEPS[
            0
        ]
    ],
    shape_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

shape_features, shape_spacings = (
    spatial_shapes_from_model(
        shape_model
    )
)

resolution_rows = []

for layer_index, (
    feature,
    spacing,
) in enumerate(
    zip(
        shape_features,
        shape_spacings,
    )
):
    original_shape = tuple(
        int(v)
        for v
        in feature.shape[
            -3:
        ]
    )

    capped, capped_spacing = (
        _cap_feature_tokens(
            feature,
            spacing,
            int(
                cfg.decoder.max_spatial_tokens
            ),
        )
    )

    capped_shape = tuple(
        int(v)
        for v
        in capped.shape[
            -3:
        ]
    )

    gt_cap, _, source_cap, internal_cap = (
        labels_at_shape(
            capped_shape
        )
    )

    token_counts = [
        int(
            np.count_nonzero(
                gt_cap
                == int(
                    gt_id
                )
            )
        )
        for gt_id in source_gt_ids
    ]

    resolution_rows.append({
        "decoder_layer": layer_index,
        "input_shape": str(
            original_shape
        ),
        "input_tokens": int(
            np.prod(
                original_shape
            )
        ),
        "capped_shape": str(
            capped_shape
        ),
        "capped_tokens": int(
            np.prod(
                capped_shape
            )
        ),
        "capped_spacing_z_um": float(
            capped_spacing[
                0, 0
            ].detach().cpu()
        ),
        "capped_spacing_y_um": float(
            capped_spacing[
                0, 1
            ].detach().cpu()
        ),
        "capped_spacing_x_um": float(
            capped_spacing[
                0, 2
            ].detach().cpu()
        ),
        "source9_tokens": int(
            source_cap.sum()
        ),
        "source9_internal_boundary_tokens": int(
            internal_cap.sum()
        ),
        "min_tokens_per_gt_cell": int(
            min(
                token_counts
            )
        ),
        "median_tokens_per_gt_cell": float(
            np.median(
                token_counts
            )
        ),
        "max_tokens_per_gt_cell": int(
            max(
                token_counts
            )
        ),
        "gt_cells_with_zero_tokens": int(
            np.sum(
                np.asarray(
                    token_counts
                )
                == 0
            )
        ),
        "gt_cells_with_leq1_token": int(
            np.sum(
                np.asarray(
                    token_counts
                )
                <= 1
            )
        ),
    })

resolution_df = pd.DataFrame(
    resolution_rows
)

resolution_df.to_csv(
    RUN_DIR
    / "decoder_resolution_audit.csv",
    index=False,
)

display(
    resolution_df
)

del (
    shape_model,
    shape_features,
)
gc.collect()
torch.cuda.empty_cache()

# 4. Parameter drift across curriculum boundaries

This tells us what actually changed numerically between saved stages.

A module with ~zero parameter drift cannot be the direct cause of an activation change in the pure-spatial forward.

In [ ]:
GROUP_PREFIXES = {
    "acquisition": (
        "acquisition.",
    ),
    "spatial_encoder": (
        "encoder.",
    ),
    "spatial_decoder": (
        "decoder.",
    ),
    "dense_heads": (
        "dense_heads.",
    ),
    "detection_graph": (
        "graph_encoder.",
    ),
    "history": (
        "history_encoder.",
        "history_fusion.",
    ),
    "tracklet_temporal": (
        "tracklet_pooler.",
        "temporal_builder.",
    ),
    "cr1_cr2": (
        "cr1.",
        "cr2.",
    ),
    "query_builder": (
        "query_builder.",
    ),
    "query_decoder": (
        "query_decoder.",
    ),
    "native_head": (
        "native_mask_head.",
    ),
}

def load_raw_checkpoint(
    path,
):
    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    state = {
        key: value.detach().cpu()
        for key, value
        in payload[
            "model"
        ].items()
    }

    step = int(
        payload.get(
            "step",
            -1,
        )
    )

    del payload

    return state, step

def relative_group_change(
    state_a,
    state_b,
    prefixes,
):
    num = 0.0
    den = 0.0
    count = 0

    for key, a in (
        state_a.items()
    ):
        if not key.startswith(
            prefixes
        ):
            continue

        c = state_b.get(
            key
        )

        if (
            c is None
            or c.shape
            != a.shape
            or not torch.is_floating_point(
                a
            )
        ):
            continue

        aa = a.float()
        cc = c.float()

        num += float(
            (
                cc - aa
            ).square().sum()
        )

        den += float(
            aa.square().sum()
        )

        count += int(
            aa.numel()
        )

    if count == 0:
        return float("nan")

    return math.sqrt(
        num
        / max(
            den,
            1e-30,
        )
    )

parameter_rows = []
MODULE_STATES = {}

previous_state = None
previous_step = None

for step in AVAILABLE_STEPS:
    state, payload_step = (
        load_raw_checkpoint(
            CHECKPOINTS[
                step
            ]
        )
    )

    if payload_step != step:
        print(
            "WARNING checkpoint metadata:",
            step,
            "payload step:",
            payload_step,
        )

    MODULE_STATES[
        step
    ] = {
        "dense_heads": {
            key[
                len(
                    "dense_heads."
                ):
            ]: value.clone()
            for key, value
            in state.items()
            if key.startswith(
                "dense_heads."
            )
        },
        "query_builder": {
            key[
                len(
                    "query_builder."
                ):
            ]: value.clone()
            for key, value
            in state.items()
            if key.startswith(
                "query_builder."
            )
        },
    }

    if previous_state is not None:
        for group, prefixes in (
            GROUP_PREFIXES.items()
        ):
            parameter_rows.append({
                "from_step": previous_step,
                "to_step": step,
                "transition": (
                    f"{previous_step}->{step}"
                ),
                "group": group,
                "relative_l2_delta": (
                    relative_group_change(
                        previous_state,
                        state,
                        prefixes,
                    )
                ),
            })

        del previous_state

    previous_state = state
    previous_step = step

del previous_state
gc.collect()

parameter_df = pd.DataFrame(
    parameter_rows
)

parameter_df.to_csv(
    RUN_DIR
    / "parameter_drift.csv",
    index=False,
)

display(
    parameter_df
)

if len(
    parameter_df
):
    display(
        parameter_df.pivot(
            index="group",
            columns="transition",
            values="relative_l2_delta",
        )
    )

# 5. Pure-spatial checkpoint sweep

For every checkpoint:

```text
spatial input
→ acquisition
→ SpatialEncoder
→ SpatialDecoder
→ DenseAuxiliaryHeads
```

No temporal graph. No CR1. No CR2.

We cache only what is needed for later forward-only swaps.

In [ ]:
def binary_auc(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    pos = (
        labels == 1
    )
    neg = (
        labels == 0
    )

    if (
        pos.sum() == 0
        or neg.sum() == 0
    ):
        return float("nan")

    ranks = rankdata(
        scores
    )

    n_pos = int(
        pos.sum()
    )
    n_neg = int(
        neg.sum()
    )

    return float(
        (
            ranks[
                pos
            ].sum()
            - n_pos
            * (
                n_pos + 1
            )
            / 2
        )
        / (
            n_pos
            * n_neg
        )
    )

def hard_dice_np(
    probability,
    target_mask,
    threshold=0.5,
):
    pred = (
        np.asarray(
            probability
        )
        >= threshold
    )

    target_mask = np.asarray(
        target_mask
    ).astype(bool)

    den = (
        pred.sum()
        + target_mask.sum()
    )

    if den == 0:
        return 1.0

    return float(
        2
        * np.logical_and(
            pred,
            target_mask,
        ).sum()
        / den
    )

def source_boundary_metrics(
    boundary_probability,
):
    shape = (
        boundary_probability.shape
    )

    _, _, source, internal = (
        labels_at_shape(
            shape
        )
    )

    positive = (
        boundary_probability[
            internal
        ]
    )

    negative = (
        boundary_probability[
            source
            & ~internal
        ]
    )

    if (
        len(
            positive
        ) == 0
        or len(
            negative
        ) == 0
    ):
        return {
            "internal_boundary_auc": float(
                "nan"
            ),
            "internal_boundary_mean": float(
                "nan"
            ),
            "source_nonboundary_mean": float(
                "nan"
            ),
            "internal_boundary_recall_0p5": float(
                "nan"
            ),
            "internal_boundary_voxels": int(
                len(
                    positive
                )
            ),
        }

    scores = np.concatenate(
        [
            positive,
            negative,
        ]
    )

    labels = np.concatenate(
        [
            np.ones(
                len(
                    positive
                ),
                dtype=np.int64,
            ),
            np.zeros(
                len(
                    negative
                ),
                dtype=np.int64,
            ),
        ]
    )

    return {
        "internal_boundary_auc": (
            binary_auc(
                scores,
                labels,
            )
        ),
        "internal_boundary_mean": float(
            positive.mean()
        ),
        "source_nonboundary_mean": float(
            negative.mean()
        ),
        "internal_boundary_recall_0p5": float(
            (
                positive
                >= 0.5
            ).mean()
        ),
        "internal_boundary_voxels": int(
            len(
                positive
            )
        ),
    }

def center_peak_metrics(
    center_probability,
    spacing_um,
):
    probability = torch.from_numpy(
        center_probability
    ).float()

    shape = tuple(
        int(v)
        for v
        in probability.shape
    )

    _, _, source, _ = (
        labels_at_shape(
            shape
        )
    )

    source_t = torch.from_numpy(
        source
    ).bool()

    local_max = (
        probability
        >= F.max_pool3d(
            probability[
                None, None
            ],
            kernel_size=3,
            stride=1,
            padding=1,
        )[0, 0]
    ) & source_t

    candidate = torch.nonzero(
        local_max.flatten(),
        as_tuple=False,
    ).flatten()

    scores = probability.flatten()[
        candidate
    ]

    order = torch.argsort(
        scores,
        descending=True,
    )

    candidate = candidate[
        order
    ]
    scores = scores[
        order
    ]

    spacing = torch.as_tensor(
        spacing_um
    ).float()

    coords = (
        feature_grid_coordinates_um(
            shape,
            spacing[
                None
            ],
            relative_to_center=True,
        )[0]
        .float()
        .cpu()
    )

    selected = []
    selected_scores = []

    min_distance_um = (
        0.45
        * dref_um
    )

    for idx, score in zip(
        candidate.tolist(),
        scores.tolist(),
    ):
        coord = coords[
            idx
        ]

        if selected:
            distance = (
                torch.linalg.vector_norm(
                    torch.stack(
                        selected
                    )
                    - coord[
                        None
                    ],
                    dim=-1,
                )
            )

            if float(
                distance.min()
            ) < min_distance_um:
                continue

        selected.append(
            coord
        )
        selected_scores.append(
            float(
                score
            )
        )

        if len(
            selected
        ) >= max(
            2
            * len(
                source_gt_ids
            ),
            len(
                source_gt_ids
            )
            + 4,
        ):
            break

    selected_coords = (
        torch.stack(
            selected
        )
        if selected
        else torch.zeros(
            (
                0,
                3,
            )
        )
    )

    k = len(
        source_gt_ids
    )

    top = selected_coords[
        :k
    ]

    if len(
        top
    ):
        distance = torch.cdist(
            top.float(),
            source_gt_centers_um.float(),
        ).numpy()

        r, c = (
            linear_sum_assignment(
                distance
            )
        )

        matched = distance[
            r,
            c,
        ]
    else:
        matched = np.zeros(
            0,
            dtype=float,
        )

    return {
        "center_peak_candidates": int(
            len(
                selected_coords
            )
        ),
        "center_topK_available": int(
            len(
                top
            )
        ),
        "center_match_mean_um": (
            float(
                np.mean(
                    matched
                )
            )
            if len(
                matched
            )
            else float(
                "nan"
            )
        ),
        "center_match_median_um": (
            float(
                np.median(
                    matched
                )
            )
            if len(
                matched
            )
            else float(
                "nan"
            )
        ),
        "centers_within_0p5_dref": (
            int(
                np.sum(
                    matched
                    <= 0.5
                    * dref_um
                )
            )
            if len(
                matched
            )
            else 0
        ),
    }

def feature_cell_separability(
    feature_cpu,
):
    feature = (
        feature_cpu.float()
    )

    shape = tuple(
        int(v)
        for v
        in feature.shape[
            -3:
        ]
    )

    gt, _, _, _ = (
        labels_at_shape(
            shape
        )
    )

    gt_t = torch.from_numpy(
        gt
    )

    centroids = []
    vectors = []
    class_labels = []

    for class_index, gt_id in (
        enumerate(
            source_gt_ids
        )
    ):
        mask = (
            gt_t
            == int(
                gt_id
            )
        )

        voxels = (
            feature
            .permute(
                1,
                2,
                3,
                0,
            )[
                mask
            ]
        )

        if len(
            voxels
        ) == 0:
            continue

        if len(
            voxels
        ) > 256:
            index = torch.linspace(
                0,
                len(
                    voxels
                )
                - 1,
                256,
            ).long()

            voxels = voxels[
                index
            ]

        centroids.append(
            voxels.mean(
                dim=0
            )
        )

        vectors.append(
            voxels
        )

        class_labels.append(
            torch.full(
                (
                    len(
                        voxels
                    ),
                ),
                class_index,
                dtype=torch.long,
            )
        )

    if len(
        centroids
    ) < 2:
        return {
            "cell_centroid_cosine": float(
                "nan"
            ),
            "nearest_cell_centroid_accuracy": float(
                "nan"
            ),
            "cell_feature_chance": float(
                "nan"
            ),
        }

    centroids = torch.stack(
        centroids
    )

    x = torch.cat(
        vectors
    )

    y = torch.cat(
        class_labels
    )

    cosine = (
        F.normalize(
            centroids,
            dim=-1,
        )
        @ F.normalize(
            centroids,
            dim=-1,
        ).T
    )

    off = ~torch.eye(
        len(
            centroids
        ),
        dtype=torch.bool,
    )

    prediction = torch.cdist(
        x,
        centroids,
    ).argmin(
        dim=-1
    )

    return {
        "cell_centroid_cosine": float(
            cosine[
                off
            ].mean()
        ),
        "nearest_cell_centroid_accuracy": float(
            (
                prediction
                == y
            ).float().mean()
        ),
        "cell_feature_chance": (
            1.0
            / len(
                centroids
            )
        ),
    }

def boundary_linear_probe(
    feature_cpu,
    *,
    max_per_class=2000,
):
    feature = (
        feature_cpu.float()
    )

    shape = tuple(
        int(v)
        for v
        in feature.shape[
            -3:
        ]
    )

    _, _, source, internal = (
        labels_at_shape(
            shape
        )
    )

    source_t = torch.from_numpy(
        source
    ).bool()

    internal_t = torch.from_numpy(
        internal
    ).bool()

    flat_feature = (
        feature
        .permute(
            1,
            2,
            3,
            0,
        )
    )

    pos = flat_feature[
        internal_t
    ]

    neg = flat_feature[
        source_t
        & ~internal_t
    ]

    n = min(
        len(
            pos
        ),
        len(
            neg
        ),
        int(
            max_per_class
        ),
    )

    if n < 10:
        return {
            "boundary_probe_auc": float(
                "nan"
            ),
            "boundary_probe_samples_per_class": int(
                n
            ),
        }

    gen = torch.Generator()
    gen.manual_seed(
        SEED
    )

    pos = pos[
        torch.randperm(
            len(
                pos
            ),
            generator=gen,
        )[
            :n
        ]
    ]

    neg = neg[
        torch.randperm(
            len(
                neg
            ),
            generator=gen,
        )[
            :n
        ]
    ]

    split = max(
        5,
        int(
            0.6
            * n
        ),
    )

    train = torch.cat(
        [
            pos[
                :split
            ],
            neg[
                :split
            ],
        ]
    )

    mean = train.mean(
        dim=0
    )

    std = train.std(
        dim=0
    ).clamp_min(
        1e-5
    )

    pos_train = (
        pos[
            :split
        ]
        - mean
    ) / std

    neg_train = (
        neg[
            :split
        ]
        - mean
    ) / std

    direction = (
        pos_train.mean(
            dim=0
        )
        - neg_train.mean(
            dim=0
        )
    )

    test = torch.cat(
        [
            pos[
                split:
            ],
            neg[
                split:
            ],
        ]
    )

    labels = np.concatenate(
        [
            np.ones(
                len(
                    pos[
                        split:
                    ]
                ),
                dtype=np.int64,
            ),
            np.zeros(
                len(
                    neg[
                        split:
                    ]
                ),
                dtype=np.int64,
            ),
        ]
    )

    standardized = (
        test - mean
    ) / std

    scores = (
        standardized
        @ direction
    ).numpy()

    return {
        "boundary_probe_auc": (
            binary_auc(
                scores,
                labels,
            )
        ),
        "boundary_probe_samples_per_class": int(
            n
        ),
    }

In [ ]:
@torch.no_grad()
def pure_spatial_forward(
    model,
):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        acquisition = (
            model.acquisition(
                b["spacing_um"],
                b["dref_um"],
            )
        )

        pyramid = (
            model.encoder(
                b[
                    "spatial_inputs"
                ],
                b["spacing_um"],
                acquisition,
                b.get(
                    "spatial_padding_mask"
                ),
            )
        )

        e3 = (
            pyramid.features[
                3
            ]
        )

        # Pure spatial path: no CR1.
        e2 = (
            model.decoder
            .decode_to_e2(
                e3,
                pyramid,
                acquisition,
            )
        )

        # Pure spatial path: no CR2.
        d1, d0, _ = (
            model.decoder
            .decode_from_e2(
                e2,
                pyramid,
                acquisition,
            )
        )

        dense = (
            model.dense_heads(
                d0
            )
        )

    return {
        "e3": e3,
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "dense": dense,
        "spacings": [
            pyramid.spacings_um[
                3
            ],
            pyramid.spacings_um[
                2
            ],
            pyramid.spacings_um[
                1
            ],
            pyramid.spacings_um[
                0
            ],
        ],
    }

SPATIAL_CACHE = {}
checkpoint_metric_rows = []
feature_probe_rows = []

foreground_target = (
    gt_label_map_cpu
    > 0
)

sweep_start = time.perf_counter()

for step in AVAILABLE_STEPS:
    print(
        "\nCheckpoint",
        step,
        STAGE_BY_STEP[
            step
        ],
    )

    model = StirNet(
        cfg
    ).to(
        device
    )

    load_checkpoint(
        CHECKPOINTS[
            step
        ],
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    spatial = (
        pure_spatial_forward(
            model
        )
    )

    boundary_prob = (
        spatial[
            "dense"
        ][
            "boundary_logits"
        ][0, 0]
        .sigmoid()
        .float()
        .cpu()
        .numpy()
    )

    center_prob = (
        spatial[
            "dense"
        ][
            "center_heatmap_logits"
        ][0, 0]
        .sigmoid()
        .float()
        .cpu()
        .numpy()
    )

    foreground_prob = (
        spatial[
            "dense"
        ][
            "foreground_logits"
        ][0, 0]
        .sigmoid()
        .float()
        .cpu()
        .numpy()
    )

    fg_target_ds = (
        resize_label_map_nearest(
            torch.from_numpy(
                foreground_target.astype(
                    np.int32
                )
            ),
            foreground_prob.shape,
        ).bool().numpy()
    )

    boundary_stats = (
        source_boundary_metrics(
            boundary_prob
        )
    )

    center_stats = (
        center_peak_metrics(
            center_prob,
            spatial[
                "spacings"
            ][3][0]
            .float()
            .cpu()
            .numpy(),
        )
    )

    checkpoint_metric_rows.append({
        "step": step,
        "stage": STAGE_BY_STEP[
            step
        ],
        "foreground_hard_dice": (
            hard_dice_np(
                foreground_prob,
                fg_target_ds,
            )
        ),
        **boundary_stats,
        **center_stats,
        "forward_seconds": float(
            time.perf_counter()
            - start
        ),
        "peak_cuda_gib": float(
            torch.cuda.max_memory_allocated()
            / 1024**3
        ),
    })

    for level in [
        "e2",
        "d1",
        "d0",
    ]:
        feature_cpu = (
            spatial[
                level
            ][0]
            .detach()
            .cpu()
            .to(
                torch.float16
            )
        )

        cell_stats = (
            feature_cell_separability(
                feature_cpu
            )
        )

        probe_stats = (
            boundary_linear_probe(
                feature_cpu
            )
        )

        feature_probe_rows.append({
            "step": step,
            "stage": STAGE_BY_STEP[
                step
            ],
            "feature_level": level.upper(),
            **cell_stats,
            **probe_stats,
        })

    # Cache only E3/E2/D1 for query swaps.
    SPATIAL_CACHE[
        step
    ] = {
        "e3": (
            spatial[
                "e3"
            ][0:1]
            .detach()
            .cpu()
            .to(
                torch.float16
            )
        ),
        "e2": (
            spatial[
                "e2"
            ][0:1]
            .detach()
            .cpu()
            .to(
                torch.float16
            )
        ),
        "d1": (
            spatial[
                "d1"
            ][0:1]
            .detach()
            .cpu()
            .to(
                torch.float16
            )
        ),
        "d0": (
            spatial[
                "d0"
            ][0:1]
            .detach()
            .cpu()
            .to(
                torch.float16
            )
        ),
        "spacings": [
            item.detach()
            .cpu()
            .float()
            for item
            in spatial[
                "spacings"
            ]
        ],
    }

    del (
        model,
        spatial,
    )
    gc.collect()
    torch.cuda.empty_cache()

checkpoint_metrics_df = pd.DataFrame(
    checkpoint_metric_rows
)

feature_probe_df = pd.DataFrame(
    feature_probe_rows
)

checkpoint_metrics_df.to_csv(
    RUN_DIR
    / "checkpoint_spatial_metrics.csv",
    index=False,
)

feature_probe_df.to_csv(
    RUN_DIR
    / "frozen_feature_probes.csv",
    index=False,
)

print(
    "\nTotal pure-spatial sweep:",
    round(
        time.perf_counter()
        - sweep_start,
        2,
    ),
    "s",
)

display(
    checkpoint_metrics_df
)

display(
    feature_probe_df
)

# 6. Activation drift between checkpoints

Parameter drift says **what changed**. Activation drift says **what effect those changes had on the actual sample**.

This uses the cached pure-spatial features only.

In [ ]:
def relative_tensor_delta(
    a,
    c,
    chunk=2_000_000,
):
    if a.shape != c.shape:
        return float(
            "nan"
        )

    aa = a.reshape(
        -1
    )
    cc = c.reshape(
        -1
    )

    num = 0.0
    den = 0.0

    for start in range(
        0,
        aa.numel(),
        chunk,
    ):
        end = min(
            start + chunk,
            aa.numel(),
        )

        x = aa[
            start:end
        ].float()

        y = cc[
            start:end
        ].float()

        num += float(
            (
                y - x
            ).square().sum()
        )

        den += float(
            x.square().sum()
        )

    return math.sqrt(
        num
        / max(
            den,
            1e-30,
        )
    )

activation_rows = []

for previous, current in zip(
    AVAILABLE_STEPS[
        :-1
    ],
    AVAILABLE_STEPS[
        1:
    ],
):
    for level in [
        "e3",
        "e2",
        "d1",
        "d0",
    ]:
        activation_rows.append({
            "from_step": previous,
            "to_step": current,
            "transition": (
                f"{previous}->{current}"
            ),
            "feature_level": (
                level.upper()
            ),
            "relative_l2_activation_delta": (
                relative_tensor_delta(
                    SPATIAL_CACHE[
                        previous
                    ][
                        level
                    ],
                    SPATIAL_CACHE[
                        current
                    ][
                        level
                    ],
                )
            ),
        })

activation_df = pd.DataFrame(
    activation_rows
)

activation_df.to_csv(
    RUN_DIR
    / "activation_drift.csv",
    index=False,
)

display(
    activation_df
)

if len(
    activation_df
):
    display(
        activation_df.pivot(
            index="feature_level",
            columns="transition",
            values="relative_l2_activation_delta",
        )
    )

# 7. Dense-head × spatial-feature swap matrix

The dense boundary and center heads are only `1×1×1` convolutions.

That lets us cheaply isolate:

```text
spatial feature checkpoint
        ×
dense-head checkpoint
```

without rerunning the CNN.

Interpretation:

- **early features + late head still good** → head is not the main cause;
- **late features + early head still bad** → spatial representation degraded;
- **early features + late head bad, early features + early head good** → dense head drift/training is implicated.

In [ ]:
def apply_dense_head(
    feature_cpu,
    state,
    name,
):
    weight = state[
        f"{name}.weight"
    ].float()

    bias = state[
        f"{name}.bias"
    ].float()

    # 1x1x1 convolution on CPU.
    logits = (
        torch.einsum(
            "oc,bczyx->bozyx",
            weight[
                :, :, 0, 0, 0
            ],
            feature_cpu.float(),
        )
        + bias[
            None,
            :,
            None,
            None,
            None,
        ]
    )

    return logits[
        0, 0
    ]

dense_swap_rows = []

for feature_step in AVAILABLE_STEPS:
    feature = (
        SPATIAL_CACHE[
            feature_step
        ][
            "d0"
        ]
    )

    spacing = (
        SPATIAL_CACHE[
            feature_step
        ][
            "spacings"
        ][3][0]
        .numpy()
    )

    for head_step in AVAILABLE_STEPS:
        state = (
            MODULE_STATES[
                head_step
            ][
                "dense_heads"
            ]
        )

        boundary_prob = (
            apply_dense_head(
                feature,
                state,
                "boundary",
            )
            .sigmoid()
            .numpy()
        )

        center_prob = (
            apply_dense_head(
                feature,
                state,
                "center",
            )
            .sigmoid()
            .numpy()
        )

        boundary_stats = (
            source_boundary_metrics(
                boundary_prob
            )
        )

        center_stats = (
            center_peak_metrics(
                center_prob,
                spacing,
            )
        )

        dense_swap_rows.append({
            "feature_step": feature_step,
            "head_step": head_step,
            "boundary_auc": (
                boundary_stats[
                    "internal_boundary_auc"
                ]
            ),
            "boundary_recall_0p5": (
                boundary_stats[
                    "internal_boundary_recall_0p5"
                ]
            ),
            "centers_within_0p5_dref": (
                center_stats[
                    "centers_within_0p5_dref"
                ]
            ),
            "center_match_median_um": (
                center_stats[
                    "center_match_median_um"
                ]
            ),
        })

dense_swap_df = pd.DataFrame(
    dense_swap_rows
)

dense_swap_df.to_csv(
    RUN_DIR
    / "dense_head_feature_swap_matrix.csv",
    index=False,
)

print("Boundary AUC matrix")
display(
    dense_swap_df.pivot(
        index="feature_step",
        columns="head_step",
        values="boundary_auc",
    )
)

print(
    "Center recovery (# GT centers within 0.5 dref)"
)
display(
    dense_swap_df.pivot(
        index="feature_step",
        columns="head_step",
        values="centers_within_0p5_dref",
    )
)

# 8. Identify the best pure-spatial checkpoint

This checkpoint will be used to test the query subsystem under the most favorable spatial representation already learned by the model.

The selection uses source-9 internal-boundary AUC, with the frozen D0 boundary-probe AUC as a tiebreaker.

In [ ]:
d0_probe = (
    feature_probe_df[
        feature_probe_df[
            "feature_level"
        ]
        == "D0"
    ][
        [
            "step",
            "boundary_probe_auc",
        ]
    ]
)

selection = (
    checkpoint_metrics_df.merge(
        d0_probe,
        on="step",
        how="left",
    )
)

selection[
    "selection_score"
] = (
    selection[
        "internal_boundary_auc"
    ].fillna(
        -1
    )
    + 0.05
    * selection[
        "boundary_probe_auc"
    ].fillna(
        -1
    )
)

BEST_SPATIAL_STEP = int(
    selection.sort_values(
        "selection_score",
        ascending=False,
    ).iloc[
        0
    ][
        "step"
    ]
)

FINAL_STEP = max(
    AVAILABLE_STEPS
)

print(
    "Best pure-spatial checkpoint:",
    BEST_SPATIAL_STEP,
    STAGE_BY_STEP[
        BEST_SPATIAL_STEP
    ],
)
print(
    "Final available checkpoint:",
    FINAL_STEP,
    STAGE_BY_STEP[
        FINAL_STEP
    ],
)

display(
    selection.sort_values(
        "selection_score",
        ascending=False,
    )
)

# 9. Query-decoder token-cap + mask-space expressivity probe

This is a particularly important diagnostic.

The coarse mask at each decoder layer is:

```text
query mask vector · spatial mask-feature vector
```

Therefore, even with a perfect query, a cell can only be isolated if the mask-feature space itself permits a suitable linear mask vector.

We compute a **GT-fitted oracle linear mask-vector upper bound**:

- use frozen spatial features;
- project them through the checkpoint's real `mask_feature_proj`;
- fit a tiny ridge linear classifier for each source-9 GT cell;
- measure how well that feature space could express the individual masks.

This is diagnostic only.

We compare:

1. **uncapped D1 crop** — preserves the available spatial lattice;
2. **real capped D1** — exactly the global `_cap_feature_tokens` rule used by the query decoder.

If uncapped expressivity is good and capped expressivity collapses, `max_spatial_tokens` is directly implicated.

In [ ]:
def gt_fitted_mask_vector_probe(
    mask_feature,
    gt_labels,
    current_labels,
    *,
    ridge=1e-2,
):
    # mask_feature [C,Z,Y,X], CPU float.
    feature = (
        mask_feature.float()
        .permute(
            1,
            2,
            3,
            0,
        )
    )

    source = (
        current_labels
        == SOURCE_ID
    )

    source_t = torch.from_numpy(
        source
    ).bool()

    x = feature[
        source_t
    ]

    if len(
        x
    ) < 10:
        return {
            "oracle_mask_probe_dice_mean": float(
                "nan"
            ),
            "oracle_mask_probe_dice_median": float(
                "nan"
            ),
            "probe_source_tokens": int(
                len(
                    x
                )
            ),
        }

    # Add bias.
    X = torch.cat(
        [
            x,
            torch.ones(
                (
                    len(
                        x
                    ),
                    1,
                )
            ),
        ],
        dim=-1,
    )

    identity = torch.eye(
        X.shape[
            1
        ]
    )

    identity[
        -1,
        -1,
    ] = 0.0

    gram = (
        X.T
        @ X
        + ridge
        * identity
    )

    dices = []

    source_gt = (
        gt_labels[
            source
        ]
    )

    for gt_id in source_gt_ids:
        y = torch.from_numpy(
            (
                source_gt
                == int(
                    gt_id
                )
            ).astype(
                np.float32
            )
        )

        # Map {0,1} to {-1,+1}.
        target_vector = (
            2.0
            * y
            - 1.0
        )

        rhs = (
            X.T
            @ target_vector
        )

        weight = torch.linalg.solve(
            gram,
            rhs,
        )

        logits = (
            X
            @ weight
        )

        probability = torch.sigmoid(
            logits
        )

        intersection = (
            probability
            * y
        ).sum()

        dice = (
            2
            * intersection
            + 1e-6
        ) / (
            probability.sum()
            + y.sum()
            + 1e-6
        )

        dices.append(
            float(
                dice
            )
        )

    return {
        "oracle_mask_probe_dice_mean": float(
            np.mean(
                dices
            )
        ),
        "oracle_mask_probe_dice_median": float(
            np.median(
                dices
            )
        ),
        "probe_source_tokens": int(
            len(
                x
            )
        ),
    }

def crop_to_source(
    feature,
    gt_labels,
    current_labels,
    margin=2,
):
    source = (
        current_labels
        == SOURCE_ID
    )

    coords = np.where(
        source
    )

    if not len(
        coords[
            0
        ]
    ):
        raise RuntimeError(
            "Source component vanished at this resolution."
        )

    low = [
        max(
            int(
                axis.min()
            )
            - margin,
            0,
        )
        for axis in coords
    ]

    high = [
        min(
            int(
                axis.max()
            )
            + 1
            + margin,
            feature.shape[
                index + 1
            ],
        )
        for index, axis
        in enumerate(
            coords
        )
    ]

    slices = tuple(
        slice(
            lo,
            hi,
        )
        for lo, hi
        in zip(
            low,
            high,
        )
    )

    return (
        feature[
            (
                slice(
                    None
                ),
                *slices,
            )
        ],
        gt_labels[
            slices
        ],
        current_labels[
            slices
        ],
    )

QUERY_STEPS = [
    step
    for step in AVAILABLE_STEPS
    if step >= 70
]

mask_probe_rows = []

if QUERY_STEPS:
    spatial_cpu = (
        SPATIAL_CACHE[
            BEST_SPATIAL_STEP
        ]
    )

    for decoder_step in QUERY_STEPS:
        qmodel = StirNet(
            cfg
        ).to(
            device
        )

        load_checkpoint(
            CHECKPOINTS[
                decoder_step
            ],
            qmodel,
            optimizer=None,
            scheduler=None,
            scaler=None,
            map_location="cpu",
            strict=True,
            migrate_history=True,
        )

        qmodel.eval()

        d1 = (
            spatial_cpu[
                "d1"
            ].to(
                device
            )
        )

        d1_spacing = (
            spatial_cpu[
                "spacings"
            ][2].to(
                device
            )
        )

        # Uncapped D1: crop first because mask_feature_proj is 1x1x1.
        d1_shape = tuple(
            int(v)
            for v
            in d1.shape[
                -3:
            ]
        )

        gt_d1, current_d1, _, _ = (
            labels_at_shape(
                d1_shape
            )
        )

        d1_crop, gt_crop, current_crop = (
            crop_to_source(
                d1[
                    0
                ],
                gt_d1,
                current_d1,
                margin=2,
            )
        )

        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            uncapped_mask_feature = (
                qmodel.query_decoder
                .mask_feature_proj[
                    2
                ](
                    d1_crop[
                        None
                    ]
                )[0]
                .detach()
                .float()
                .cpu()
            )

        uncapped_probe = (
            gt_fitted_mask_vector_probe(
                uncapped_mask_feature,
                gt_crop,
                current_crop,
            )
        )

        # Real capped final decoder level.
        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            capped_d1, capped_spacing = (
                _cap_feature_tokens(
                    d1,
                    d1_spacing,
                    int(
                        cfg.decoder.max_spatial_tokens
                    ),
                )
            )

            capped_mask_feature = (
                qmodel.query_decoder
                .mask_feature_proj[
                    2
                ](
                    capped_d1
                )[0]
                .detach()
                .float()
                .cpu()
            )

        capped_shape = tuple(
            int(v)
            for v
            in capped_mask_feature.shape[
                -3:
            ]
        )

        gt_cap, current_cap, _, _ = (
            labels_at_shape(
                capped_shape
            )
        )

        capped_probe = (
            gt_fitted_mask_vector_probe(
                capped_mask_feature,
                gt_cap,
                current_cap,
            )
        )

        mask_probe_rows.append({
            "spatial_step": (
                BEST_SPATIAL_STEP
            ),
            "decoder_step": (
                decoder_step
            ),
            "mode": "uncapped_D1",
            **uncapped_probe,
        })

        mask_probe_rows.append({
            "spatial_step": (
                BEST_SPATIAL_STEP
            ),
            "decoder_step": (
                decoder_step
            ),
            "mode": "capped_D1_real_decoder",
            **capped_probe,
        })

        del (
            qmodel,
            d1,
        )
        gc.collect()
        torch.cuda.empty_cache()

mask_probe_df = pd.DataFrame(
    mask_probe_rows
)

mask_probe_df.to_csv(
    RUN_DIR
    / "mask_feature_expressivity_probe.csv",
    index=False,
)

display(
    mask_probe_df
)

# 10. Pure-spatial query decomposition across query checkpoints

We now hard-disable temporal memory and test the actual QueryBuilder + QueryDecoder.

Two views:

### Diagonal trajectory
Each query checkpoint uses its own same-step pure spatial features.

### Best-spatial hybrid matrix
The best spatial checkpoint is held fixed while QueryBuilder and QueryDecoder weights are independently swapped across query stages.

Every configuration is evaluated both with:

- baseline source-centroid references;
- oracle GT centers for source-9 seeded queries.

If **all** oracle configurations remain poor on the best spatial features, the query/mask formulation itself is strongly implicated.

In [ ]:
def empty_temporal(
    model,
):
    d = int(
        model.cfg.temporal.d_model
    )

    edge_dim = int(
        model.cfg.temporal.hypothesis_edge_dim
    )

    return TemporalState(
        tokens=torch.zeros(
            (
                0,
                d,
            ),
            device=device,
            dtype=torch.float16,
        ),
        ref_um=torch.zeros(
            (
                0,
                3,
            ),
            device=device,
            dtype=torch.float32,
        ),
        ref_cellscale=torch.zeros(
            (
                0,
                3,
            ),
            device=device,
            dtype=torch.float32,
        ),
        salience=torch.zeros(
            (
                0,
                1,
            ),
            device=device,
            dtype=torch.float16,
        ),
        reliability=torch.zeros(
            (
                0,
                1,
            ),
            device=device,
            dtype=torch.float16,
        ),
        status=torch.zeros(
            (
                0,
                1,
            ),
            device=device,
            dtype=torch.float32,
        ),
        edge_index=torch.zeros(
            (
                2,
                0,
            ),
            device=device,
            dtype=torch.long,
        ),
        edge_attr=torch.zeros(
            (
                0,
                edge_dim,
            ),
            device=device,
            dtype=torch.float32,
        ),
        batch_index=torch.zeros(
            (
                0,
            ),
            device=device,
            dtype=torch.long,
        ),
    )

def disable_temporal_query_memory(
    model,
):
    model.query_builder.component_memory_enabled = (
        False
    )

    for layer in (
        model.query_decoder.layers
    ):
        layer.query_memory_enabled = (
            False
        )

def source9_query_indices(
    qstate,
):
    source = (
        qstate.source_instance_ids[
            0
        ]
    )

    qtype = (
        qstate.query_types[
            0
        ]
    )

    mask = (
        (
            source
            == SOURCE_ID
        )
        & (
            (
                qtype
                == QUERY_PRIMARY
            )
            | (
                qtype
                == QUERY_SPLIT
            )
        )
    )

    return torch.nonzero(
        mask,
        as_tuple=False,
    ).flatten()

def with_oracle_centers(
    qstate,
):
    indices = (
        source9_query_indices(
            qstate
        )
    )

    refs = (
        qstate.references_cellscale
        .clone()
    )

    sorted_gt = (
        source_gt_centers_um[
            torch.argsort(
                source_gt_centers_um[
                    :, 0
                ]
                * 1e6
                + source_gt_centers_um[
                    :, 1
                ]
                * 1e3
                + source_gt_centers_um[
                    :, 2
                ]
            )
        ]
    )

    count = min(
        len(
            indices
        ),
        len(
            sorted_gt
        ),
    )

    refs[
        0,
        indices[
            :count
        ],
    ] = (
        sorted_gt[
            :count
        ].to(
            refs.device,
            dtype=refs.dtype,
        )
        / b["dref_um"][
            0
        ].to(
            refs.device,
            dtype=refs.dtype,
        )
    )

    return replace(
        qstate,
        references_cellscale=refs,
    )

def source9_query_metrics(
    qstate,
    outputs,
):
    final = outputs[
        -1
    ]

    indices = (
        source9_query_indices(
            qstate
        )
    )

    logits = (
        final[
            "coarse_mask_logits"
        ][
            0,
            indices,
        ]
        .float()
    )

    shape = tuple(
        int(v)
        for v
        in logits.shape[
            -3:
        ]
    )

    gt, _, _, _ = (
        labels_at_shape(
            shape
        )
    )

    gt_t = torch.from_numpy(
        gt
    ).to(
        logits.device
    )

    pred = (
        logits.sigmoid()
        .flatten(
            1
        )
    )

    target_masks = torch.stack([
        (
            gt_t
            == int(
                gt_id
            )
        ).float()
        for gt_id
        in source_gt_ids
    ]).flatten(
        1
    )

    intersection = (
        pred[
            :, None, :
        ]
        * target_masks[
            None, :, :
        ]
    ).sum(
        dim=-1
    )

    dice = (
        2
        * intersection
        + 1e-6
    ) / (
        pred.sum(
            dim=-1
        )[
            :, None
        ]
        + target_masks.sum(
            dim=-1
        )[
            None, :
        ]
        + 1e-6
    )

    r, c = (
        linear_sum_assignment(
            -dice.detach()
            .cpu()
            .numpy()
        )
    )

    assigned = dice[
        r,
        c,
    ]

    pair_intersection = (
        pred
        @ pred.T
    )

    pred_sum = pred.sum(
        dim=-1
    )

    pair_dice = (
        2
        * pair_intersection
        + 1e-6
    ) / (
        pred_sum[
            :, None
        ]
        + pred_sum[
            None, :
        ]
        + 1e-6
    )

    off = ~torch.eye(
        len(
            indices
        ),
        dtype=torch.bool,
        device=logits.device,
    )

    centers_um = (
        qstate.references_cellscale[
            0,
            indices,
        ]
        .float()
        * b["dref_um"][
            0
        ].float()
    )

    center_distance = (
        torch.cdist(
            centers_um,
            source_gt_centers_um.to(
                centers_um.device
            ),
        )
    )

    cr, cg = (
        linear_sum_assignment(
            center_distance.detach()
            .cpu()
            .numpy()
        )
    )

    return {
        "source9_query_count": int(
            len(
                indices
            )
        ),
        "best_match_soft_dice_mean": float(
            assigned.mean()
            .detach()
            .cpu()
        ),
        "best_match_soft_dice_median": float(
            assigned.median()
            .detach()
            .cpu()
        ),
        "sibling_mask_pair_dice_mean": float(
            pair_dice[
                off
            ].mean()
            .detach()
            .cpu()
        ),
        "center_to_gt_mean_um": float(
            center_distance[
                cr,
                cg,
            ].mean()
            .detach()
            .cpu()
        ),
        "center_to_gt_median_um": float(
            center_distance[
                cr,
                cg,
            ].median()
            .detach()
            .cpu()
        ),
    }

def install_query_builder_state(
    model,
    builder_step,
):
    state = MODULE_STATES[
        builder_step
    ][
        "query_builder"
    ]

    missing, unexpected = (
        model.query_builder
        .load_state_dict(
            state,
            strict=False,
        )
    )

    # Missing keys are allowed only for migrated/new temporal-memory additions.
    serious_missing = [
        key
        for key in missing
        if "temporal_fusion"
        not in key
    ]

    if serious_missing or unexpected:
        raise RuntimeError(
            f"Query-builder state mismatch. missing={serious_missing}, "
            f"unexpected={unexpected}"
        )

@torch.no_grad()
def evaluate_query_configuration(
    *,
    spatial_step,
    builder_step,
    decoder_step,
    oracle_centers,
):
    model = StirNet(
        cfg
    ).to(
        device
    )

    # Decoder weights and all shared model defaults come from decoder_step.
    load_checkpoint(
        CHECKPOINTS[
            decoder_step
        ],
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    if (
        builder_step
        != decoder_step
    ):
        install_query_builder_state(
            model,
            builder_step,
        )

    disable_temporal_query_memory(
        model
    )

    model.eval()

    cache = (
        SPATIAL_CACHE[
            spatial_step
        ]
    )

    e3 = cache[
        "e3"
    ].to(
        device
    )

    e2 = cache[
        "e2"
    ].to(
        device
    )

    d1 = cache[
        "d1"
    ].to(
        device
    )

    spacings = [
        item.to(
            device
        )
        for item
        in cache[
            "spacings"
        ][
            :3
        ]
    ]

    temporal = (
        empty_temporal(
            model
        )
    )

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        qstate = (
            model.query_builder(
                e2,
                spacings[
                    1
                ],
                b[
                    "instance_labels"
                ],
                b[
                    "instance_features"
                ],
                b[
                    "instance_ids"
                ],
                b[
                    "instance_batch"
                ],
                b[
                    "instance_centroids_um"
                ],
                b[
                    "dref_um"
                ],
                temporal,
            )
        )

        if oracle_centers:
            qstate = (
                with_oracle_centers(
                    qstate
                )
            )

        qstate, outputs = (
            model.query_decoder(
                qstate,
                [
                    e3,
                    e2,
                    d1,
                ],
                spacings,
                b[
                    "instance_labels"
                ],
                b[
                    "dref_um"
                ],
                temporal=None,
            )
        )

    metrics = (
        source9_query_metrics(
            qstate,
            outputs,
        )
    )

    del (
        model,
        e3,
        e2,
        d1,
    )
    gc.collect()
    torch.cuda.empty_cache()

    return metrics

In [ ]:
query_rows = []

if QUERY_STEPS:
    query_start = time.perf_counter()

    # A. Diagonal trajectory.
    for step in QUERY_STEPS:
        for oracle in [
            False,
            True,
        ]:
            metrics = (
                evaluate_query_configuration(
                    spatial_step=step,
                    builder_step=step,
                    decoder_step=step,
                    oracle_centers=oracle,
                )
            )

            query_rows.append({
                "experiment": "diagonal",
                "spatial_step": step,
                "builder_step": step,
                "decoder_step": step,
                "oracle_centers": oracle,
                **metrics,
            })

    # B. Builder x decoder matrix on best spatial features.
    for builder_step in QUERY_STEPS:
        for decoder_step in QUERY_STEPS:
            for oracle in [
                False,
                True,
            ]:
                metrics = (
                    evaluate_query_configuration(
                        spatial_step=BEST_SPATIAL_STEP,
                        builder_step=builder_step,
                        decoder_step=decoder_step,
                        oracle_centers=oracle,
                    )
                )

                query_rows.append({
                    "experiment": "best_spatial_hybrid",
                    "spatial_step": (
                        BEST_SPATIAL_STEP
                    ),
                    "builder_step": builder_step,
                    "decoder_step": decoder_step,
                    "oracle_centers": oracle,
                    **metrics,
                })

    print(
        "Query swap sweep:",
        round(
            time.perf_counter()
            - query_start,
            2,
        ),
        "s",
    )

query_df = pd.DataFrame(
    query_rows
)

query_df.to_csv(
    RUN_DIR
    / "query_checkpoint_swap_matrix.csv",
    index=False,
)

display(
    query_df
)

if len(
    query_df
):
    oracle_matrix = (
        query_df[
            (
                query_df[
                    "experiment"
                ]
                == "best_spatial_hybrid"
            )
            & (
                query_df[
                    "oracle_centers"
                ]
            )
        ]
    )

    if len(
        oracle_matrix
    ):
        print(
            "Best-spatial + ORACLE centers: soft Dice matrix"
        )

        display(
            oracle_matrix.pivot(
                index="builder_step",
                columns="decoder_step",
                values="best_match_soft_dice_mean",
            )
        )

# 11. Automatic localization report

The report intentionally gives **evidence statements**, not a speculative architecture rewrite.

The strongest conclusions come from combinations of independent probes.

In [ ]:
def row_for_step(
    dataframe,
    step,
):
    rows = dataframe[
        dataframe[
            "step"
        ]
        == step
    ]

    return (
        rows.iloc[
            0
        ]
        if len(
            rows
        )
        else None
    )

EARLIEST_STEP = min(
    AVAILABLE_STEPS
)

earliest = row_for_step(
    checkpoint_metrics_df,
    EARLIEST_STEP,
)

final = row_for_step(
    checkpoint_metrics_df,
    FINAL_STEP,
)

best = row_for_step(
    checkpoint_metrics_df,
    BEST_SPATIAL_STEP,
)

evidence = []

# 1. Decoder resolution.
final_layer_resolution = (
    resolution_df[
        resolution_df[
            "decoder_layer"
        ]
        == 2
    ].iloc[
        0
    ]
)

if (
    final_layer_resolution[
        "gt_cells_with_leq1_token"
    ]
    > 0
    or final_layer_resolution[
        "source9_internal_boundary_tokens"
    ]
    < len(
        source_gt_ids
    )
):
    evidence.append(
        "HIGH PRIORITY: the real query-decoder token cap leaves extremely sparse "
        "source-9 spatial support at the final decoder level. The decoder may be "
        "discarding the resolution needed for touching-cell separation before attention."
    )

# 2. Did dense spatial evidence ever become good?
best_auc = float(
    best[
        "internal_boundary_auc"
    ]
)

final_auc = float(
    final[
        "internal_boundary_auc"
    ]
)

if best_auc >= 0.70:
    evidence.append(
        f"The pure spatial pathway DID learn useful internal-boundary evidence at step "
        f"{BEST_SPATIAL_STEP} (AUC={best_auc:.3f})."
    )

    if (
        final_auc
        < best_auc
        - 0.10
    ):
        evidence.append(
            f"That boundary evidence later degraded to AUC={final_auc:.3f} by step "
            f"{FINAL_STEP}. The spatial failure is at least partly training-stage regression, "
            "not simply an incapable CNN."
        )
else:
    evidence.append(
        f"No checkpoint has strong source-9 internal-boundary AUC "
        f"(best={best_auc:.3f}). The spatial dense pathway never cleanly learned this easy merge."
    )

# 3. Frozen feature probe.
best_probe_row = (
    feature_probe_df[
        (
            feature_probe_df[
                "step"
            ]
            == BEST_SPATIAL_STEP
        )
        & (
            feature_probe_df[
                "feature_level"
            ]
            == "D0"
        )
    ]
)

if len(
    best_probe_row
):
    best_probe_auc = float(
        best_probe_row.iloc[
            0
        ][
            "boundary_probe_auc"
        ]
    )

    if (
        best_probe_auc
        >= 0.75
        and best_auc
        < 0.65
    ):
        evidence.append(
            "D0 frozen features contain substantially more internal-boundary information "
            "than the trained boundary head extracts. Dense-head optimization/readout is implicated."
        )
    elif best_probe_auc < 0.65:
        evidence.append(
            "Even a GT-fitted frozen D0 boundary probe is weak. This supports a genuine "
            "spatial feature-representation problem rather than only a bad 1x1 boundary head."
        )

# 4. Dense head vs feature matrix.
if (
    EARLIEST_STEP
    in AVAILABLE_STEPS
    and FINAL_STEP
    in AVAILABLE_STEPS
):
    def swap_value(
        feature_step,
        head_step,
        column,
    ):
        rows = dense_swap_df[
            (
                dense_swap_df[
                    "feature_step"
                ]
                == feature_step
            )
            & (
                dense_swap_df[
                    "head_step"
                ]
                == head_step
            )
        ]

        return (
            float(
                rows.iloc[
                    0
                ][
                    column
                ]
            )
            if len(
                rows
            )
            else float(
                "nan"
            )
        )

    early_early = swap_value(
        EARLIEST_STEP,
        EARLIEST_STEP,
        "boundary_auc",
    )

    early_final = swap_value(
        EARLIEST_STEP,
        FINAL_STEP,
        "boundary_auc",
    )

    final_early = swap_value(
        FINAL_STEP,
        EARLIEST_STEP,
        "boundary_auc",
    )

    final_final = swap_value(
        FINAL_STEP,
        FINAL_STEP,
        "boundary_auc",
    )

    if (
        early_early >= 0.70
        and early_final >= early_early - 0.05
        and final_early < early_early - 0.10
    ):
        evidence.append(
            "Dense-head swap localizes degradation primarily to D0 spatial FEATURES: "
            "the late head still works on early features, while the early head cannot rescue late features."
        )

    elif (
        early_early >= 0.70
        and early_final < early_early - 0.10
        and final_early >= final_final + 0.10
    ):
        evidence.append(
            "Dense-head swap implicates the dense boundary HEAD itself: late head weights "
            "lose information that earlier head weights can still extract."
        )

# 5. Mask expressivity / cap.
if len(
    mask_probe_df
):
    uncapped_best = float(
        mask_probe_df[
            mask_probe_df[
                "mode"
            ]
            == "uncapped_D1"
        ][
            "oracle_mask_probe_dice_mean"
        ].max()
    )

    capped_best = float(
        mask_probe_df[
            mask_probe_df[
                "mode"
            ]
            == "capped_D1_real_decoder"
        ][
            "oracle_mask_probe_dice_mean"
        ].max()
    )

    if (
        uncapped_best
        > capped_best
        + 0.10
    ):
        evidence.append(
            f"Mask-space expressivity drops strongly after real token capping "
            f"(oracle probe {uncapped_best:.3f} → {capped_best:.3f}). "
            "`max_spatial_tokens` is a direct candidate bottleneck."
        )
    elif uncapped_best < 0.30:
        evidence.append(
            f"Even the uncapped D1 mask-feature space has low GT-fitted instance expressivity "
            f"(best oracle probe={uncapped_best:.3f}). The mask-feature representation itself is weak."
        )
    elif capped_best >= uncapped_best - 0.05:
        evidence.append(
            "Capping does not materially reduce the GT-fitted mask-feature upper bound; "
            "the primary query failure lies elsewhere."
        )

# 6. Query subsystem.
if len(
    query_df
):
    hybrid_oracle = (
        query_df[
            (
                query_df[
                    "experiment"
                ]
                == "best_spatial_hybrid"
            )
            & (
                query_df[
                    "oracle_centers"
                ]
            )
        ]
    )

    if len(
        hybrid_oracle
    ):
        best_oracle_query = float(
            hybrid_oracle[
                "best_match_soft_dice_mean"
            ].max()
        )

        evidence.append(
            f"Best query-builder/decoder combination on the best spatial checkpoint, "
            f"with oracle centers, reaches source-9 coarse soft Dice={best_oracle_query:.4f}."
        )

        if best_oracle_query < 0.05:
            evidence.append(
                "CRITICAL: no learned query-stage checkpoint can form useful source-9 coarse masks "
                "even with the best available spatial features AND oracle centers. "
                "The query/mask-decoding pathway has an independent structural/training failure."
            )

# 7. Parameter/activation freeze consistency.
for previous, current in zip(
    AVAILABLE_STEPS[
        :-1
    ],
    AVAILABLE_STEPS[
        1:
    ],
):
    if (
        previous
        >= 50
        and current
        <= 80
    ):
        spatial_drift = parameter_df[
            (
                parameter_df[
                    "from_step"
                ]
                == previous
            )
            & (
                parameter_df[
                    "to_step"
                ]
                == current
            )
            & (
                parameter_df[
                    "group"
                ].isin(
                    [
                        "spatial_encoder",
                        "spatial_decoder",
                    ]
                )
            )
        ][
            "relative_l2_delta"
        ]

        if (
            len(
                spatial_drift
            )
            and float(
                spatial_drift.max()
            )
            < 1e-8
        ):
            activation_drift = activation_df[
                (
                    activation_df[
                        "from_step"
                    ]
                    == previous
                )
                & (
                    activation_df[
                        "to_step"
                    ]
                    == current
                )
                & (
                    activation_df[
                        "feature_level"
                    ]
                    .isin(
                        [
                            "E2",
                            "D1",
                            "D0",
                        ]
                    )
                )
            ][
                "relative_l2_activation_delta"
            ]

            if (
                len(
                    activation_drift
                )
                and float(
                    activation_drift.max()
                )
                > 1e-5
            ):
                evidence.append(
                    f"INCONSISTENCY: spatial parameters are frozen across {previous}->{current} "
                    "but pure-spatial activations changed measurably. Re-check checkpoint provenance "
                    "or diagnostic determinism before architectural conclusions."
                )

report = {
    "available_steps": (
        AVAILABLE_STEPS
    ),
    "best_spatial_step": (
        BEST_SPATIAL_STEP
    ),
    "final_step": (
        FINAL_STEP
    ),
    "evidence": evidence,
}

print(
    "="
    * 78
)
print(
    "NOTEBOOK 20 — SPATIAL FAILURE LOCALIZATION"
)
print(
    "="
    * 78
)

for index, statement in (
    enumerate(
        evidence,
        start=1,
    )
):
    print(
        f"{index}. {statement}"
    )

with (
    RUN_DIR
    / "localization_report.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=float,
    )

# 12. Compact trajectory plots

In [ ]:
trajectory_columns = [
    (
        "internal_boundary_auc",
        "Source-9 internal boundary AUC",
    ),
    (
        "centers_within_0p5_dref",
        "Source-9 center peaks within 0.5 dref",
    ),
    (
        "foreground_hard_dice",
        "Global foreground hard Dice",
    ),
]

for column, title in (
    trajectory_columns
):
    ax = (
        checkpoint_metrics_df
        .set_index(
            "step"
        )[
            column
        ]
        .plot(
            marker="o",
            figsize=(
                7,
                3.5,
            ),
        )
    )

    ax.set_title(
        title
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    plt.tight_layout()
    plt.show()

for level in [
    "E2",
    "D1",
    "D0",
]:
    subset = (
        feature_probe_df[
            feature_probe_df[
                "feature_level"
            ]
            == level
        ]
    )

    ax = (
        subset
        .set_index(
            "step"
        )[
            "boundary_probe_auc"
        ]
        .plot(
            marker="o",
            figsize=(
                7,
                3.5,
            ),
        )
    )

    ax.set_title(
        f"{level} frozen boundary-probe AUC"
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    plt.tight_layout()
    plt.show()

# 13. Save timing and artifact manifest

In [ ]:
manifest = {
    "checkpoint_run": str(
        CHECKPOINT_RUN
    ),
    "available_steps": (
        AVAILABLE_STEPS
    ),
    "best_spatial_step": (
        BEST_SPATIAL_STEP
    ),
    "artifacts": [
        path.name
        for path in sorted(
            RUN_DIR.glob(
                "*"
            )
        )
    ],
}

with (
    RUN_DIR
    / "manifest.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

print(
    json.dumps(
        manifest,
        indent=2,
    )
)

gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA cache cleared."
)

# How to act on the result

Notebook 20 is intended to narrow the next change without another long training run.

## If the spatial representation was good early and later degraded

Do **not** redesign the CNN.

Locate the curriculum transition and protect the spatial representation:

- freezing;
- LR isolation;
- gradient isolation;
- loss balancing;
- stage-specific regression tests.

## If D0 features are good but the trained dense head is bad

Fix dense-head optimization/supervision rather than the backbone.

## If uncapped mask-feature expressivity is good but capped expressivity is bad

The likely bottleneck is **query-decoder spatial token compression**.

The next targeted change should test a higher-resolution/local/chunked spatial attention path, not temporal routing.

## If even uncapped mask features cannot linearly isolate individual cells

The mask-feature representation itself is inadequate for instance decomposition.

Investigate:

- mask feature projection;
- instance-aware spatial supervision;
- feature resolution/context;
- whether spatial features encode only foreground class and not instance identity.

## If the best spatial checkpoint is good but every oracle-center query configuration is bad

The spatial CNN is not the main blocker.

Focus on:

```text
QueryBuilder
→ QueryCrossAttention
→ mask_feature_proj
→ query mask embedding
→ coarse mask objective
```

## If no checkpoint ever develops strong internal-boundary/center evidence

Then the failure is genuinely upstream in spatial training:

```text
SpatialEncoder / SpatialDecoder
+ dense targets / losses
+ resolution / receptive field
```

At that point we should inspect and change the spatial objective/architecture directly rather than spending more effort on temporal reasoning.

# STIR-Net V1 — Notebook 20 additional spatial-debugging cells

**Append/run these cells after the already executed Notebook 20.**

Nothing from the existing Notebook 20 is repeated. These cells assume the same kernel still contains its variables, caches, and helper functions.

The new diagnostics add five missing pieces before we move to the implementation fix:

1. recover the **exact step-30 spatial-only checkpoint** used to seed Notebook 15;
2. isolate what happened specifically during **step 30 → 50 temporal-dense training**;
3. audit whether the **spatial supervision itself** is adequate for touching-cell decomposition;
4. trace the spatial decoder **operation by operation** to find where E2 separation information disappears;
5. compare **mask-space expressivity and actual query decoding** while increasing the spatial-token budget.

There is **no optimizer and no training loop** in these additional cells.

In [ ]:
from scipy import ndimage as ndi

_REQUIRED_NB20_NAMES = [
    "REPO_ROOT",
    "RUN_DIR",
    "cfg",
    "device",
    "b",
    "batch",
    "target",
    "gt_label_map_cpu",
    "current_labels_cpu",
    "source_mask_cpu",
    "source_gt_ids",
    "source_gt_centers_um",
    "dref_um",
    "SPATIAL_CACHE",
    "CHECKPOINTS",
    "AVAILABLE_STEPS",
    "checkpoint_metrics_df",
    "feature_probe_df",
    "GROUP_PREFIXES",
    "relative_group_change",
    "load_raw_checkpoint",
    "pure_spatial_forward",
    "hard_dice_np",
    "source_boundary_metrics",
    "center_peak_metrics",
    "feature_cell_separability",
    "boundary_linear_probe",
    "labels_at_shape",
    "gt_fitted_mask_vector_probe",
    "crop_to_source",
    "empty_temporal",
    "disable_temporal_query_memory",
    "source9_query_indices",
    "with_oracle_centers",
    "source9_query_metrics",
]

_missing = [
    name
    for name in _REQUIRED_NB20_NAMES
    if name not in globals()
]

if _missing:
    raise RuntimeError(
        "These cells are a continuation of the already executed Notebook 20.\n"
        "The following Notebook-20 variables/functions are missing:\n"
        + "\n".join(f"  - {name}" for name in _missing)
        + "\n\nRun the existing Notebook 20 through its diagnostic helper/caching cells, "
          "then execute this continuation."
    )

print("Notebook-20 continuation preflight OK.")
print("Existing available steps:", AVAILABLE_STEPS)

# A. Recover the exact step-30 spatial checkpoint

Notebook 13 and Notebook 15 both identify the same file:

```text
runs/stirnet/first_overfit/12_staged_same_sample/checkpoint_spatial_dense.pt
```

Notebook 15 then loads that model state and begins its first optimizer step at curriculum step 30. Therefore the Notebook-15 step-50 checkpoint is a direct continuation of this step-30 spatial state.

This gives us the missing causal comparison:

```text
step 30: end spatial_dense
   ↓ 20 temporal-dense steps
step 50: end temporal_dense
```

In [ ]:
NB12_STEP30 = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

if not NB12_STEP30.exists():
    raise FileNotFoundError(
        "The exact Notebook-12 spatial checkpoint is missing:\n"
        f"{NB12_STEP30}\n\n"
        "Notebook 13/15 used this exact path."
    )

_step30_payload = torch.load(
    NB12_STEP30,
    map_location="cpu",
    weights_only=False,
)

STEP30_PAYLOAD_STEP = int(
    _step30_payload.get("step", -1)
)

print("Step-30 checkpoint:", NB12_STEP30)
print("Saved payload step :", STEP30_PAYLOAD_STEP)

if STEP30_PAYLOAD_STEP != 30:
    raise RuntimeError(
        f"Expected checkpoint step 30, found {STEP30_PAYLOAD_STEP}."
    )

STEP30_RAW_STATE = {
    key: value.detach().cpu()
    for key, value
    in _step30_payload["model"].items()
}

del _step30_payload

step30_model = StirNet(cfg).to(device)

loaded30 = load_checkpoint(
    NB12_STEP30,
    step30_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

print("Migrated load step:", loaded30.get("step"))
print(
    "Migration notes:",
    len(
        loaded30.get(
            "temporal_migration",
            loaded30.get("history_migration", []),
        )
    ),
)

## A1. Run the exact Notebook-20 pure-spatial diagnostic at step 30

In [ ]:
torch.cuda.reset_peak_memory_stats()

with torch.no_grad():
    STEP30_SPATIAL = pure_spatial_forward(
        step30_model
    )

boundary30 = (
    STEP30_SPATIAL["dense"]["boundary_logits"][0, 0]
    .sigmoid()
    .float()
    .cpu()
    .numpy()
)

center30 = (
    STEP30_SPATIAL["dense"]["center_heatmap_logits"][0, 0]
    .sigmoid()
    .float()
    .cpu()
    .numpy()
)

foreground30 = (
    STEP30_SPATIAL["dense"]["foreground_logits"][0, 0]
    .sigmoid()
    .float()
    .cpu()
    .numpy()
)

fg_target30 = resize_label_map_nearest(
    torch.from_numpy(
        (gt_label_map_cpu > 0).astype(np.int32)
    ),
    foreground30.shape,
).bool().numpy()

boundary30_stats = source_boundary_metrics(
    boundary30
)

center30_stats = center_peak_metrics(
    center30,
    STEP30_SPATIAL["spacings"][3][0]
    .detach()
    .float()
    .cpu()
    .numpy(),
)

step30_metric = {
    "step": 30,
    "stage": "spatial_dense_end",
    "foreground_hard_dice": hard_dice_np(
        foreground30,
        fg_target30,
    ),
    **boundary30_stats,
    **center30_stats,
}

step30_feature_rows = []

for level in ("e2", "d1", "d0"):
    feature_cpu = (
        STEP30_SPATIAL[level][0]
        .detach()
        .cpu()
        .to(torch.float16)
    )

    step30_feature_rows.append({
        "step": 30,
        "stage": "spatial_dense_end",
        "feature_level": level.upper(),
        **feature_cell_separability(
            feature_cpu
        ),
        **boundary_linear_probe(
            feature_cpu
        ),
    })

step30_feature_df = pd.DataFrame(
    step30_feature_rows
)

SPATIAL_CACHE[30] = {
    "e3": (
        STEP30_SPATIAL["e3"][0:1]
        .detach().cpu().to(torch.float16)
    ),
    "e2": (
        STEP30_SPATIAL["e2"][0:1]
        .detach().cpu().to(torch.float16)
    ),
    "d1": (
        STEP30_SPATIAL["d1"][0:1]
        .detach().cpu().to(torch.float16)
    ),
    "d0": (
        STEP30_SPATIAL["d0"][0:1]
        .detach().cpu().to(torch.float16)
    ),
    "spacings": [
        item.detach().cpu().float()
        for item in STEP30_SPATIAL["spacings"]
    ],
}

STEP30_DENSE_CPU = {
    key: value.detach().float().cpu()
    for key, value
    in STEP30_SPATIAL["dense"].items()
}

step30_vs_existing_df = pd.concat(
    [
        pd.DataFrame([step30_metric]),
        checkpoint_metrics_df[
            checkpoint_metrics_df["step"].isin(
                [50, 70, 80, 85]
            )
        ],
    ],
    ignore_index=True,
    sort=False,
)

step30_vs_existing_df.to_csv(
    RUN_DIR
    / "additional_step30_checkpoint_trajectory.csv",
    index=False,
)

display(
    step30_vs_existing_df[
        [
            "step",
            "stage",
            "foreground_hard_dice",
            "internal_boundary_auc",
            "internal_boundary_recall_0p5",
            "centers_within_0p5_dref",
            "center_match_median_um",
        ]
    ]
)

display(step30_feature_df)

del STEP30_SPATIAL
gc.collect()
torch.cuda.empty_cache()

# B. What changed specifically during step 30 → 50?

We compare:

1. **step-30 pure spatial**;
2. **step-50 pure spatial** with CR bypassed;
3. **step-50 CR-active dense path**, matching temporal-dense topology.

We also measure parameter drift from the exact step-30 ancestor to step 50.

In [ ]:
if 50 not in CHECKPOINTS:
    raise RuntimeError(
        "Notebook 20 did not locate the Notebook-15 step-50 checkpoint."
    )

STEP50_PATH = CHECKPOINTS[50]
STEP50_RAW_STATE, STEP50_PAYLOAD_STEP = (
    load_raw_checkpoint(
        STEP50_PATH
    )
)

if STEP50_PAYLOAD_STEP != 50:
    raise RuntimeError(
        f"Expected step-50 payload, found {STEP50_PAYLOAD_STEP}."
    )

drift_30_50_rows = []

for group, prefixes in GROUP_PREFIXES.items():
    drift_30_50_rows.append({
        "transition": "30->50",
        "group": group,
        "relative_l2_delta": (
            relative_group_change(
                STEP30_RAW_STATE,
                STEP50_RAW_STATE,
                prefixes,
            )
        ),
    })

drift_30_50_df = pd.DataFrame(
    drift_30_50_rows
).sort_values(
    "relative_l2_delta",
    ascending=False,
)

drift_30_50_df.to_csv(
    RUN_DIR
    / "additional_parameter_drift_30_to_50.csv",
    index=False,
)

display(drift_30_50_df)

In [ ]:
@torch.no_grad()
def temporal_dense_spatial_forward(
    model,
):
    # Dense spatial output with real temporal + CR1 + CR2, but no queries.
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        acquisition = model.acquisition(
            b["spacing_um"],
            b["dref_um"],
        )

        pyramid = model.encoder(
            b["spatial_inputs"],
            b["spacing_um"],
            acquisition,
            b.get("spatial_padding_mask"),
        )

        temporal = model._build_temporal(
            b["graph_x"],
            b["graph_edge_index"],
            b["graph_edge_attr"],
            b["tracklet_id"],
            b["temporal_ref_um"],
            b["temporal_status"],
            b["hypothesis_edge_index"],
            b["hypothesis_edge_attr"],
            b["temporal_batch"],
            b["dref_um"],
            node_instance_grid=b.get("node_instance_grid"),
            node_history_valid=b.get("node_history_valid"),
            history_support=b.get("history_support"),
            history_support_valid=b.get("history_support_valid"),
            history_support_dt=b.get("history_support_dt"),
            history_support_center_um=b.get("history_support_center_um"),
            history_support_extent_um=b.get("history_support_extent_um"),
            best_current_component_id=b.get("best_current_component_id"),
            best_component_overlap=b.get("best_component_overlap"),
            second_best_component_overlap=b.get("second_best_component_overlap"),
            node_observed_ref_um=b.get("node_observed_ref_um"),
            node_time_offset=b.get("node_time_offset"),
            node_ids=b.get("node_ids"),
        )

        e3, temporal = model.cr1(
            pyramid.features[3],
            pyramid.spacings_um[3],
            temporal,
            b["dref_um"],
            acquisition,
            (
                pyramid.padding_masks[3]
                if pyramid.padding_masks
                else None
            ),
        )

        e2 = model.decoder.decode_to_e2(
            e3,
            pyramid,
            acquisition,
        )

        e2, temporal = model.cr2(
            e2,
            pyramid.spacings_um[2],
            temporal,
            b["dref_um"],
            acquisition,
            (
                pyramid.padding_masks[2]
                if pyramid.padding_masks
                else None
            ),
        )

        d1, d0, _ = (
            model.decoder.decode_from_e2(
                e2,
                pyramid,
                acquisition,
            )
        )

        dense = model.dense_heads(
            d0
        )

    return {
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "dense": dense,
        "spacings": [
            pyramid.spacings_um[2],
            pyramid.spacings_um[1],
            pyramid.spacings_um[0],
        ],
    }


def compact_dense_metrics(
    dense,
    spacing_d0,
    label,
):
    boundary_probability = (
        dense["boundary_logits"][0, 0]
        .sigmoid()
        .float()
        .cpu()
        .numpy()
    )

    center_probability = (
        dense["center_heatmap_logits"][0, 0]
        .sigmoid()
        .float()
        .cpu()
        .numpy()
    )

    return {
        "path": label,
        **source_boundary_metrics(
            boundary_probability
        ),
        **center_peak_metrics(
            center_probability,
            spacing_d0[
                0
            ].detach()
            .float()
            .cpu()
            .numpy(),
        ),
    }


step50_model = StirNet(cfg).to(
    device
)

load_checkpoint(
    STEP50_PATH,
    step50_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

with torch.no_grad():
    step50_pure = pure_spatial_forward(
        step50_model
    )

    step50_cr = (
        temporal_dense_spatial_forward(
            step50_model
        )
    )

STEP50_PURE_DENSE_CPU = {
    key: value.detach()
    .float().cpu()
    for key, value
    in step50_pure[
        "dense"
    ].items()
}

STEP50_CR_DENSE_CPU = {
    key: value.detach()
    .float().cpu()
    for key, value
    in step50_cr[
        "dense"
    ].items()
}

comparison_30_50 = pd.DataFrame([
    {
        "path": "step30_pure_spatial",
        **boundary30_stats,
        **center30_stats,
    },
    compact_dense_metrics(
        step50_pure["dense"],
        step50_pure["spacings"][3],
        "step50_pure_spatial",
    ),
    compact_dense_metrics(
        step50_cr["dense"],
        step50_cr["spacings"][2],
        "step50_CR_active",
    ),
])

comparison_30_50.to_csv(
    RUN_DIR
    / "additional_step30_step50_pure_vs_cr.csv",
    index=False,
)

display(
    comparison_30_50[
        [
            "path",
            "internal_boundary_auc",
            "internal_boundary_recall_0p5",
            "centers_within_0p5_dref",
            "center_match_median_um",
        ]
    ]
)

del (
    step50_pure,
    step50_cr,
)
gc.collect()
torch.cuda.empty_cache()

# C. Audit the training targets themselves

Spatial-dense training has only:

```text
foreground
center heatmap
boundary
```

Foreground contains no instance identity.

These cells quantify whether internal touching-cell boundaries and crowded cell centers receive enough usable supervision.

In [ ]:
spacing_native = (
    batch["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)

training_boundary = (
    torch.as_tensor(
        target["boundary"]
    )
    .detach()
    .cpu()
    .numpy()
    > 0.5
)

internal_raw = (
    internal_instance_boundary_np(
        gt_label_map_cpu
    )
)

BOUNDARY_WIDTH_UM = 1.0

rv = np.ceil(
    BOUNDARY_WIDTH_UM
    / spacing_native
).astype(int)

zz, yy, xx = np.ogrid[
    -rv[0]:rv[0] + 1,
    -rv[1]:rv[1] + 1,
    -rv[2]:rv[2] + 1,
]

structure = (
    (zz * spacing_native[0]) ** 2
    + (yy * spacing_native[1]) ** 2
    + (xx * spacing_native[2]) ** 2
    <= BOUNDARY_WIDTH_UM ** 2
)

internal_dilated = ndi.binary_dilation(
    internal_raw,
    structure=structure,
)

internal_training_positive = (
    internal_dilated
    & training_boundary
)

outer_or_other_boundary = (
    training_boundary
    & ~internal_training_positive
)

boundary_target_audit = pd.DataFrame([
    {
        "category": "all_training_boundary_positive",
        "voxels": int(
            training_boundary.sum()
        ),
        "fraction_of_boundary_target": 1.0,
    },
    {
        "category": "internal_cell_cell_positive",
        "voxels": int(
            internal_training_positive.sum()
        ),
        "fraction_of_boundary_target": float(
            internal_training_positive.sum()
            / max(
                training_boundary.sum(),
                1,
            )
        ),
    },
    {
        "category": "outer_or_other_positive",
        "voxels": int(
            outer_or_other_boundary.sum()
        ),
        "fraction_of_boundary_target": float(
            outer_or_other_boundary.sum()
            / max(
                training_boundary.sum(),
                1,
            )
        ),
    },
    {
        "category": "source9_internal_positive",
        "voxels": int(
            (
                internal_training_positive
                & source_mask_cpu
            ).sum()
        ),
        "fraction_of_boundary_target": float(
            (
                internal_training_positive
                & source_mask_cpu
            ).sum()
            / max(
                training_boundary.sum(),
                1,
            )
        ),
    },
])

boundary_target_audit.to_csv(
    RUN_DIR
    / "additional_boundary_target_composition.csv",
    index=False,
)

display(boundary_target_audit)

In [ ]:
def resize_bool_mask(
    mask,
    shape,
):
    return (
        resize_label_map_nearest(
            torch.from_numpy(
                mask.astype(
                    np.int32
                )
            ),
            shape,
        )
        .bool()
    )


def boundary_loss_subset_audit(
    boundary_logits_cpu,
    name,
):
    logits = (
        boundary_logits_cpu[
            0, 0
        ].float()
    )

    shape = tuple(
        int(v)
        for v
        in logits.shape
    )

    train_positive = resize_bool_mask(
        training_boundary,
        shape,
    )

    internal_positive = resize_bool_mask(
        internal_training_positive,
        shape,
    ) & train_positive

    outer_positive = (
        train_positive
        & ~internal_positive
    )

    source9 = resize_bool_mask(
        source_mask_cpu,
        shape,
    )

    source9_internal = (
        internal_positive
        & source9
    )

    probability = logits.sigmoid()

    target_tensor = (
        train_positive.float()
    )

    pos_weight = torch.tensor(
        float(
            cfg.losses.boundary_pos_weight
        ),
        dtype=torch.float32,
    )

    per_voxel_bce = (
        F.binary_cross_entropy_with_logits(
            logits,
            target_tensor,
            pos_weight=pos_weight,
            reduction="none",
        )
    )

    positive_loss = (
        per_voxel_bce[
            train_positive
        ].sum()
    )

    def mean_prob(mask):
        if not bool(mask.any()):
            return float("nan")
        return float(
            probability[
                mask
            ].mean()
        )

    def recall(mask):
        if not bool(mask.any()):
            return float("nan")
        return float(
            (
                probability[
                    mask
                ]
                >= 0.5
            ).float().mean()
        )

    return {
        "model_path": name,
        "internal_mean_probability": (
            mean_prob(
                internal_positive
            )
        ),
        "internal_recall_0p5": (
            recall(
                internal_positive
            )
        ),
        "outer_mean_probability": (
            mean_prob(
                outer_positive
            )
        ),
        "outer_recall_0p5": (
            recall(
                outer_positive
            )
        ),
        "source9_internal_mean_probability": (
            mean_prob(
                source9_internal
            )
        ),
        "source9_internal_recall_0p5": (
            recall(
                source9_internal
            )
        ),
        "internal_positive_loss_share": float(
            (
                per_voxel_bce[
                    internal_positive
                ].sum()
                / positive_loss.clamp_min(
                    1e-12
                )
            )
        ),
        "source9_internal_positive_loss_share": float(
            (
                per_voxel_bce[
                    source9_internal
                ].sum()
                / positive_loss.clamp_min(
                    1e-12
                )
            )
        ),
    }


boundary_subset_df = pd.DataFrame([
    boundary_loss_subset_audit(
        STEP30_DENSE_CPU[
            "boundary_logits"
        ],
        "step30_pure_spatial",
    ),
    boundary_loss_subset_audit(
        STEP50_PURE_DENSE_CPU[
            "boundary_logits"
        ],
        "step50_pure_spatial",
    ),
    boundary_loss_subset_audit(
        STEP50_CR_DENSE_CPU[
            "boundary_logits"
        ],
        "step50_CR_active",
    ),
])

boundary_subset_df.to_csv(
    RUN_DIR
    / "additional_boundary_internal_vs_outer_behavior.csv",
    index=False,
)

display(boundary_subset_df)

## C1. Can the GT center target itself resolve the nine cells?

If the target heatmap cannot yield nine distinct peaks under the same physical NMS, the center branch is being asked to learn an already-collapsed target.

In [ ]:
target_center_heatmap = (
    torch.as_tensor(
        target[
            "center_heatmap"
        ]
    )
    .detach()
    .cpu()
    .float()
    .numpy()
)

target_center_peak_stats = (
    center_peak_metrics(
        target_center_heatmap,
        spacing_native,
    )
)

center_target_audit_df = pd.DataFrame([
    {
        "source": "GT_center_heatmap_target",
        **target_center_peak_stats,
    },
    {
        "source": "step30_prediction",
        **center30_stats,
    },
    {
        "source": "step50_pure_prediction",
        **center_peak_metrics(
            STEP50_PURE_DENSE_CPU[
                "center_heatmap_logits"
            ][0, 0]
            .sigmoid()
            .numpy(),
            spacing_native,
        ),
    },
    {
        "source": "step50_CR_active_prediction",
        **center_peak_metrics(
            STEP50_CR_DENSE_CPU[
                "center_heatmap_logits"
            ][0, 0]
            .sigmoid()
            .numpy(),
            spacing_native,
        ),
    },
])

center_target_audit_df.to_csv(
    RUN_DIR
    / "additional_center_target_adequacy.csv",
    index=False,
)

display(center_target_audit_df)

# D. Trace the spatial decoder operation by operation

For step 30 and step 50, probe:

```text
skip
→ interpolation
→ up projection
→ concat
→ 1×1 fuse
→ each residual block
```

for all three decoder stages.

In [ ]:
@torch.no_grad()
def decoder_operation_trace(
    checkpoint_path,
    step,
):
    model = StirNet(
        cfg
    ).to(
        device
    )

    load_checkpoint(
        checkpoint_path,
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    model.eval()

    rows = []

    def record(
        name,
        tensor,
    ):
        feature_cpu = (
            tensor[0]
            .detach()
            .cpu()
            .to(
                torch.float16
            )
        )

        rows.append({
            "step": int(step),
            "operation": name,
            "shape": str(
                tuple(
                    int(v)
                    for v
                    in tensor.shape[
                        -3:
                    ]
                )
            ),
            **boundary_linear_probe(
                feature_cpu
            ),
            **feature_cell_separability(
                feature_cpu
            ),
        })

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        acquisition = (
            model.acquisition(
                b[
                    "spacing_um"
                ],
                b[
                    "dref_um"
                ],
            )
        )

        pyramid = (
            model.encoder(
                b[
                    "spatial_inputs"
                ],
                b[
                    "spacing_um"
                ],
                acquisition,
                b.get(
                    "spatial_padding_mask"
                ),
            )
        )

        x = (
            pyramid.features[
                3
            ]
        )

        record(
            "encoder_E3",
            x,
        )

        stages = [
            (
                "stage_e2",
                model.decoder.stage_e2,
                pyramid.features[
                    2
                ],
            ),
            (
                "stage_e1",
                model.decoder.stage_e1,
                pyramid.features[
                    1
                ],
            ),
            (
                "stage_e0",
                model.decoder.stage_e0,
                pyramid.features[
                    0
                ],
            ),
        ]

        for stage_name, stage, skip in stages:
            record(
                f"{stage_name}.skip_input",
                skip,
            )

            interpolated = (
                F.interpolate(
                    x,
                    size=skip.shape[
                        -3:
                    ],
                    mode="trilinear",
                    align_corners=False,
                )
            )

            record(
                f"{stage_name}.interpolate",
                interpolated,
            )

            up_projected = (
                stage.up.proj(
                    interpolated
                )
            )

            record(
                f"{stage_name}.up_projection",
                up_projected,
            )

            concatenated = (
                torch.cat(
                    [
                        up_projected,
                        skip,
                    ],
                    dim=1,
                )
            )

            record(
                f"{stage_name}.concat_skip",
                concatenated,
            )

            fused = stage.fuse(
                concatenated
            )

            record(
                f"{stage_name}.fuse_1x1",
                fused,
            )

            x = fused

            for block_index, block in enumerate(
                stage.blocks
            ):
                x = block(
                    x,
                    acquisition,
                )

                record(
                    f"{stage_name}.resblock_{block_index}",
                    x,
                )

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(
        rows
    )


trace30_df = decoder_operation_trace(
    NB12_STEP30,
    30,
)

trace50_df = decoder_operation_trace(
    STEP50_PATH,
    50,
)

decoder_trace_df = pd.concat(
    [
        trace30_df,
        trace50_df,
    ],
    ignore_index=True,
)

decoder_trace_df.to_csv(
    RUN_DIR
    / "additional_decoder_operation_trace.csv",
    index=False,
)

display(
    decoder_trace_df[
        [
            "step",
            "operation",
            "shape",
            "boundary_probe_auc",
            "nearest_cell_centroid_accuracy",
            "cell_centroid_cosine",
        ]
    ]
)

In [ ]:
drop_rows = []

for step, group in decoder_trace_df.groupby(
    "step"
):
    ordered = group.reset_index(
        drop=True
    )

    for index in range(
        1,
        len(
            ordered
        ),
    ):
        before = float(
            ordered.loc[
                index - 1,
                "boundary_probe_auc",
            ]
        )

        after = float(
            ordered.loc[
                index,
                "boundary_probe_auc",
            ]
        )

        drop_rows.append({
            "step": int(
                step
            ),
            "from_operation": (
                ordered.loc[
                    index - 1,
                    "operation",
                ]
            ),
            "to_operation": (
                ordered.loc[
                    index,
                    "operation",
                ]
            ),
            "boundary_auc_before": before,
            "boundary_auc_after": after,
            "delta": after - before,
        })

decoder_drop_df = pd.DataFrame(
    drop_rows
).sort_values(
    "delta"
)

decoder_drop_df.to_csv(
    RUN_DIR
    / "additional_decoder_operation_drops.csv",
    index=False,
)

print("Largest negative boundary-probe changes:")
display(
    decoder_drop_df.head(
        12
    )
)

# E. Mask-feature expressivity at E3, E2, and D1

The actual coarse-query decoder uses:

```text
layer 0 ← E3
layer 1 ← E2
layer 2 ← D1
```

For each level, fit a GT-only diagnostic linear mask vector in the frozen projected feature space and compare uncapped versus real-capped representations.

In [ ]:
QUERY_DECODER_STEP = max(
    step
    for step in AVAILABLE_STEPS
    if step >= 70
)

query_probe_model = StirNet(
    cfg
).to(
    device
)

load_checkpoint(
    CHECKPOINTS[
        QUERY_DECODER_STEP
    ],
    query_probe_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

query_probe_model.eval()

mask_level_rows = []

spatial_probe_steps = [
    step
    for step in [
        30,
        50,
        max(
            AVAILABLE_STEPS
        ),
    ]
    if step in SPATIAL_CACHE
]

spatial_probe_steps = list(
    dict.fromkeys(
        spatial_probe_steps
    )
)

LEVELS = [
    (
        0,
        "E3",
        "e3",
        0,
    ),
    (
        1,
        "E2",
        "e2",
        1,
    ),
    (
        2,
        "D1",
        "d1",
        2,
    ),
]

for spatial_step in spatial_probe_steps:
    cache = SPATIAL_CACHE[
        spatial_step
    ]

    for (
        layer_index,
        level_name,
        cache_key,
        spacing_index,
    ) in LEVELS:
        feature_cpu = cache[
            cache_key
        ]

        spacing = cache[
            "spacings"
        ][
            spacing_index
        ]

        full_shape = tuple(
            int(v)
            for v in feature_cpu.shape[
                -3:
            ]
        )

        gt_full, current_full, _, _ = (
            labels_at_shape(
                full_shape
            )
        )

        cropped_feature, cropped_gt, cropped_current = (
            crop_to_source(
                feature_cpu[
                    0
                ],
                gt_full,
                current_full,
                margin=2,
            )
        )

        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            uncapped_projected = (
                query_probe_model
                .query_decoder
                .mask_feature_proj[
                    layer_index
                ](
                    cropped_feature[
                        None
                    ].to(
                        device
                    )
                )[
                    0
                ]
                .detach()
                .float()
                .cpu()
            )

        uncapped = (
            gt_fitted_mask_vector_probe(
                uncapped_projected,
                cropped_gt,
                cropped_current,
            )
        )

        mask_level_rows.append({
            "spatial_step": spatial_step,
            "query_decoder_step": (
                QUERY_DECODER_STEP
            ),
            "level": level_name,
            "mode": "uncapped_local_crop",
            **uncapped,
        })

        full_feature_gpu = feature_cpu.to(
            device
        )

        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            capped_feature, _ = (
                _cap_feature_tokens(
                    full_feature_gpu,
                    spacing.to(
                        device
                    ),
                    int(
                        query_probe_model
                        .query_decoder
                        .cfg
                        .max_spatial_tokens
                    ),
                )
            )

            capped_projected = (
                query_probe_model
                .query_decoder
                .mask_feature_proj[
                    layer_index
                ](
                    capped_feature
                )[
                    0
                ]
                .detach()
                .float()
                .cpu()
            )

        capped_shape = tuple(
            int(v)
            for v in capped_projected.shape[
                -3:
            ]
        )

        gt_cap, current_cap, _, _ = (
            labels_at_shape(
                capped_shape
            )
        )

        capped = (
            gt_fitted_mask_vector_probe(
                capped_projected,
                gt_cap,
                current_cap,
            )
        )

        mask_level_rows.append({
            "spatial_step": spatial_step,
            "query_decoder_step": (
                QUERY_DECODER_STEP
            ),
            "level": level_name,
            "mode": "real_global_cap",
            **capped,
        })

        del (
            full_feature_gpu,
            capped_feature,
            capped_projected,
            uncapped_projected,
        )

        gc.collect()
        torch.cuda.empty_cache()

mask_level_df = pd.DataFrame(
    mask_level_rows
)

mask_level_df.to_csv(
    RUN_DIR
    / "additional_multilevel_mask_expressivity.csv",
    index=False,
)

display(mask_level_df)

print("Uncapped oracle mask-space mean Dice:")
display(
    mask_level_df[
        mask_level_df[
            "mode"
        ]
        == "uncapped_local_crop"
    ].pivot(
        index="spatial_step",
        columns="level",
        values="oracle_mask_probe_dice_mean",
    )
)

print("Real-cap oracle mask-space mean Dice:")
display(
    mask_level_df[
        mask_level_df[
            "mode"
        ]
        == "real_global_cap"
    ].pivot(
        index="spatial_step",
        columns="level",
        values="oracle_mask_probe_dice_mean",
    )
)

del query_probe_model
gc.collect()
torch.cuda.empty_cache()

# F. Actual QueryDecoder cap sweep — source 9 only, oracle centers

No training. Only the nine source-9 seeded queries are decoded.

Caps:

```text
2,048
8,192
32,768
131,072
262,144
```

In [ ]:
from dataclasses import replace as _dc_replace

def subset_query_state(
    qstate,
    query_indices,
):
    idx = query_indices.to(
        qstate.embeddings.device
    )

    return _dc_replace(
        qstate,
        embeddings=(
            qstate.embeddings[
                :, idx
            ]
        ),
        references_cellscale=(
            qstate.references_cellscale[
                :, idx
            ]
        ),
        query_types=(
            qstate.query_types[
                :, idx
            ]
        ),
        padding_mask=(
            qstate.padding_mask[
                :, idx
            ]
        ),
        source_instance_ids=(
            qstate.source_instance_ids[
                :, idx
            ]
        ),
        temporal_salience=(
            qstate.temporal_salience[
                :, idx
            ]
        ),
        temporal_reliability=(
            qstate.temporal_reliability[
                :, idx
            ]
        ),
    )


_spatial_selection = pd.concat(
    [
        pd.DataFrame(
            [step30_metric]
        ),
        checkpoint_metrics_df,
    ],
    ignore_index=True,
    sort=False,
)

_spatial_selection = (
    _spatial_selection
    .drop_duplicates(
        subset=[
            "step"
        ],
        keep="first",
    )
)

BEST_SPATIAL_STEP_EXTENDED = int(
    _spatial_selection.sort_values(
        "internal_boundary_auc",
        ascending=False,
    ).iloc[
        0
    ][
        "step"
    ]
)

print(
    "Best spatial step for cap sweep:",
    BEST_SPATIAL_STEP_EXTENDED,
)

decoder_step = max(
    step
    for step in AVAILABLE_STEPS
    if step >= 70
)

cap_model = StirNet(
    cfg
).to(
    device
)

load_checkpoint(
    CHECKPOINTS[
        decoder_step
    ],
    cap_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

disable_temporal_query_memory(
    cap_model
)

cap_model.eval()

cap_cache = SPATIAL_CACHE[
    BEST_SPATIAL_STEP_EXTENDED
]

cap_e3 = cap_cache[
    "e3"
].to(
    device
)

cap_e2 = cap_cache[
    "e2"
].to(
    device
)

cap_d1 = cap_cache[
    "d1"
].to(
    device
)

cap_spacings = [
    item.to(
        device
    )
    for item
    in cap_cache[
        "spacings"
    ][
        :3
    ]
]

cap_temporal = empty_temporal(
    cap_model
)

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    cap_qstate_full = (
        cap_model.query_builder(
            cap_e2,
            cap_spacings[
                1
            ],
            b[
                "instance_labels"
            ],
            b[
                "instance_features"
            ],
            b[
                "instance_ids"
            ],
            b[
                "instance_batch"
            ],
            b[
                "instance_centroids_um"
            ],
            b[
                "dref_um"
            ],
            cap_temporal,
        )
    )

cap_qstate_oracle = (
    with_oracle_centers(
        cap_qstate_full
    )
)

cap_source9_indices = (
    source9_query_indices(
        cap_qstate_oracle
    )
)

cap_qstate_source9 = (
    subset_query_state(
        cap_qstate_oracle,
        cap_source9_indices,
    )
)

print(
    "Source-9 queries in cap sweep:",
    int(
        cap_qstate_source9
        .embeddings.shape[
            1
        ]
    ),
)

In [ ]:
CAPS = [
    2_048,
    8_192,
    32_768,
    131_072,
    262_144,
]

cap_rows = []

original_cap = int(
    cap_model
    .query_decoder
    .cfg
    .max_spatial_tokens
)

for cap in CAPS:
    print(
        f"\nTesting cap={cap:,} ..."
    )

    cap_model.query_decoder.cfg.max_spatial_tokens = (
        int(
            cap
        )
    )

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    try:
        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            final_q, outputs = (
                cap_model.query_decoder(
                    cap_qstate_source9,
                    [
                        cap_e3,
                        cap_e2,
                        cap_d1,
                    ],
                    cap_spacings,
                    b[
                        "instance_labels"
                    ],
                    b[
                        "dref_um"
                    ],
                    temporal=None,
                )
            )

        metrics = (
            source9_query_metrics(
                final_q,
                outputs,
            )
        )

        cap_rows.append({
            "max_spatial_tokens": int(
                cap
            ),
            "status": "ok",
            "elapsed_s": float(
                time.perf_counter()
                - start
            ),
            "peak_cuda_gib": float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
            **metrics,
        })

        print(
            "  Dice:",
            round(
                metrics[
                    "best_match_soft_dice_mean"
                ],
                6,
            ),
            "| sibling mask Dice:",
            round(
                metrics[
                    "sibling_mask_pair_dice_mean"
                ],
                6,
            ),
            "| center error:",
            round(
                metrics[
                    "center_to_gt_mean_um"
                ],
                3,
            ),
            "um",
        )

        del (
            final_q,
            outputs,
        )

    except torch.OutOfMemoryError:
        cap_rows.append({
            "max_spatial_tokens": int(
                cap
            ),
            "status": "OOM",
            "elapsed_s": float(
                time.perf_counter()
                - start
            ),
            "peak_cuda_gib": float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
        })

        print(
            "  OOM at this cap; continuing."
        )

    gc.collect()
    torch.cuda.empty_cache()

cap_model.query_decoder.cfg.max_spatial_tokens = (
    original_cap
)

actual_cap_sweep_df = pd.DataFrame(
    cap_rows
)

actual_cap_sweep_df.to_csv(
    RUN_DIR
    / "additional_actual_query_cap_sweep.csv",
    index=False,
)

display(actual_cap_sweep_df)

del (
    cap_model,
    cap_e3,
    cap_e2,
    cap_d1,
    cap_qstate_full,
    cap_qstate_oracle,
    cap_qstate_source9,
)
gc.collect()
torch.cuda.empty_cache()

# G. Final automatic localization before implementation

The report below converts the new measurements into an implementation priority list.

In [ ]:
evidence = []
priority = []

step30_auc = float(
    step30_metric[
        "internal_boundary_auc"
    ]
)

step50_pure_row = comparison_30_50[
    comparison_30_50[
        "path"
    ]
    == "step50_pure_spatial"
].iloc[
    0
]

step50_cr_row = comparison_30_50[
    comparison_30_50[
        "path"
    ]
    == "step50_CR_active"
].iloc[
    0
]

step50_pure_auc = float(
    step50_pure_row[
        "internal_boundary_auc"
    ]
)

step50_cr_auc = float(
    step50_cr_row[
        "internal_boundary_auc"
    ]
)

if step30_auc < 0.60:
    evidence.append(
        f"Spatial-only training is already weak at step 30 "
        f"(source-9 internal-boundary AUC={step30_auc:.3f}). "
        "The core failure therefore predates temporal/query/native training."
    )
    priority.append(
        "spatial_dense_supervision_or_representation"
    )
elif (
    step50_pure_auc
    < step30_auc
    - 0.10
):
    evidence.append(
        f"Spatial separation was useful at step 30 (AUC={step30_auc:.3f}) "
        f"but degraded by step 50 with CR bypassed (AUC={step50_pure_auc:.3f}). "
        "Temporal-dense optimization damaged shared spatial weights."
    )
    priority.append(
        "protect_spatial_weights_during_temporal_dense"
    )
else:
    evidence.append(
        f"Pure-spatial internal-boundary AUC changes only from "
        f"{step30_auc:.3f} to {step50_pure_auc:.3f}; temporal-dense training "
        "is not the main origin of the spatial failure."
    )

if (
    step50_cr_auc
    < step50_pure_auc
    - 0.10
):
    evidence.append(
        f"At step 50, activating CR lowers internal-boundary AUC "
        f"from {step50_pure_auc:.3f} to {step50_cr_auc:.3f}."
    )
    priority.append(
        "coreasoning_spatial_interference"
    )

internal_fraction = float(
    boundary_target_audit[
        boundary_target_audit[
            "category"
        ]
        == "internal_cell_cell_positive"
    ][
        "fraction_of_boundary_target"
    ].iloc[
        0
    ]
)

source9_fraction = float(
    boundary_target_audit[
        boundary_target_audit[
            "category"
        ]
        == "source9_internal_positive"
    ][
        "fraction_of_boundary_target"
    ].iloc[
        0
    ]
)

if internal_fraction < 0.25:
    evidence.append(
        f"Only {100*internal_fraction:.2f}% of positive boundary-target voxels "
        "are internal cell-cell boundaries. The boundary objective is dominated "
        "by outer cell/background boundaries."
    )
    priority.append(
        "instance_internal_boundary_supervision"
    )

if source9_fraction < 0.02:
    evidence.append(
        f"Source-9 internal boundaries contribute only {100*source9_fraction:.3f}% "
        "of all positive boundary voxels."
    )

target_center_recovery = int(
    center_target_audit_df[
        center_target_audit_df[
            "source"
        ]
        == "GT_center_heatmap_target"
    ][
        "centers_within_0p5_dref"
    ].iloc[
        0
    ]
)

if target_center_recovery < len(
    source_gt_ids
):
    evidence.append(
        f"Even the GT center target resolves only "
        f"{target_center_recovery}/{len(source_gt_ids)} source-9 centers."
    )
    priority.append(
        "center_target_design"
    )

valid_drops = decoder_drop_df[
    np.isfinite(
        decoder_drop_df[
            "delta"
        ]
    )
]

if len(
    valid_drops
):
    worst_drop = valid_drops.iloc[
        0
    ]

    evidence.append(
        "Largest single decoder boundary-information drop: "
        f"{worst_drop['from_operation']} → {worst_drop['to_operation']} "
        f"at step {int(worst_drop['step'])}, "
        f"ΔAUC={float(worst_drop['delta']):.3f}."
    )

    if float(
        worst_drop[
            "delta"
        ]
    ) < -0.08:
        priority.append(
            str(
                worst_drop[
                    "to_operation"
                ]
            )
        )

uncapped_mask = mask_level_df[
    mask_level_df[
        "mode"
    ]
    == "uncapped_local_crop"
]

best_uncapped_row = (
    uncapped_mask.sort_values(
        "oracle_mask_probe_dice_mean",
        ascending=False,
    ).iloc[
        0
    ]
)

best_uncapped_dice = float(
    best_uncapped_row[
        "oracle_mask_probe_dice_mean"
    ]
)

best_uncapped_level = str(
    best_uncapped_row[
        "level"
    ]
)

best_uncapped_step = int(
    best_uncapped_row[
        "spatial_step"
    ]
)

evidence.append(
    f"Best uncapped coarse mask-feature upper bound is "
    f"{best_uncapped_dice:.3f} Dice at {best_uncapped_level}, "
    f"spatial step {best_uncapped_step}."
)

if best_uncapped_dice < 0.30:
    evidence.append(
        "Even a GT-fitted mask vector cannot express strong individual-cell masks "
        "from any coarse query feature level."
    )
    priority.append(
        "instance_aware_mask_feature_representation"
    )

same_step = uncapped_mask[
    uncapped_mask[
        "spatial_step"
    ]
    == best_uncapped_step
]

if {
    "E2",
    "D1",
}.issubset(
    set(
        same_step[
            "level"
        ]
    )
):
    e2_dice = float(
        same_step[
            same_step[
                "level"
            ]
            == "E2"
        ][
            "oracle_mask_probe_dice_mean"
        ].iloc[
            0
        ]
    )

    d1_dice = float(
        same_step[
            same_step[
                "level"
            ]
            == "D1"
        ][
            "oracle_mask_probe_dice_mean"
        ].iloc[
            0
        ]
    )

    if e2_dice > d1_dice + 0.08:
        evidence.append(
            f"Mask expressivity drops from E2={e2_dice:.3f} to D1={d1_dice:.3f}."
        )
        priority.append(
            "preserve_E2_instance_information_into_D1"
        )

ok_caps = actual_cap_sweep_df[
    actual_cap_sweep_df[
        "status"
    ]
    == "ok"
].sort_values(
    "max_spatial_tokens"
)

if len(
    ok_caps
) >= 2:
    low = ok_caps.iloc[
        0
    ]
    high = ok_caps.iloc[
        -1
    ]

    low_dice = float(
        low[
            "best_match_soft_dice_mean"
        ]
    )

    high_dice = float(
        high[
            "best_match_soft_dice_mean"
        ]
    )

    cap_gain = (
        high_dice
        - low_dice
    )

    evidence.append(
        f"Actual oracle-center QueryDecoder Dice changes from "
        f"{low_dice:.4f} at {int(low['max_spatial_tokens']):,} tokens "
        f"to {high_dice:.4f} at {int(high['max_spatial_tokens']):,} tokens "
        f"(gain={cap_gain:+.4f})."
    )

    if cap_gain > 0.05:
        priority.append(
            "replace_global_token_cap_with_local_or_chunked_attention"
        )
    else:
        evidence.append(
            "Increasing the token budget does not materially rescue learned masks; "
            "token capping is not the primary current blocker."
        )

priority = list(
    dict.fromkeys(
        priority
    )
)

additional_report = {
    "step30_internal_boundary_auc": (
        step30_auc
    ),
    "step50_pure_internal_boundary_auc": (
        step50_pure_auc
    ),
    "step50_cr_internal_boundary_auc": (
        step50_cr_auc
    ),
    "internal_boundary_fraction_of_training_positive": (
        internal_fraction
    ),
    "source9_internal_fraction_of_training_positive": (
        source9_fraction
    ),
    "gt_center_target_recovery": (
        target_center_recovery
    ),
    "source9_gt_count": int(
        len(
            source_gt_ids
        )
    ),
    "best_uncapped_mask_probe_dice": (
        best_uncapped_dice
    ),
    "best_uncapped_mask_probe_level": (
        best_uncapped_level
    ),
    "priority_order": (
        priority
    ),
    "evidence": evidence,
}

print(
    "=" * 84
)
print(
    "NOTEBOOK 20 — FINAL ADDITIONAL LOCALIZATION"
)
print(
    "=" * 84
)

for index, item in enumerate(
    evidence,
    start=1,
):
    print(
        f"{index}. {item}"
    )

print(
    "\nIMPLEMENTATION PRIORITY:"
)

for index, item in enumerate(
    priority,
    start=1,
):
    print(
        f"{index}. {item}"
    )

with (
    RUN_DIR
    / "additional_final_localization_report.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        additional_report,
        handle,
        indent=2,
        default=float,
    )

# H. Save artifact manifest

After this cell, the diagnostic phase is complete. The next step should be the measured implementation fix.

In [ ]:
additional_artifacts = [
    "additional_step30_checkpoint_trajectory.csv",
    "additional_parameter_drift_30_to_50.csv",
    "additional_step30_step50_pure_vs_cr.csv",
    "additional_boundary_target_composition.csv",
    "additional_boundary_internal_vs_outer_behavior.csv",
    "additional_center_target_adequacy.csv",
    "additional_decoder_operation_trace.csv",
    "additional_decoder_operation_drops.csv",
    "additional_multilevel_mask_expressivity.csv",
    "additional_actual_query_cap_sweep.csv",
    "additional_final_localization_report.json",
]

print("Additional Notebook-20 artifacts:")
for name in additional_artifacts:
    path = RUN_DIR / name
    print(
        " ",
        "✓" if path.exists() else "✗",
        name,
    )

gc.collect()
torch.cuda.empty_cache()

print(
    "\nAdditional spatial debugging complete. "
    "No training was performed."
)